# CredResolve — Dataset Profiling and Data Forensics

## Purpose
This notebook profiles the raw CSV tables and investigates data-quality risks before business metrics are trusted. It focuses on record duplication, key consistency, missingness, payment status/references, account and borrower mappings, targeting, and timestamp-related issues.

## Approach
- Keep the original CSV files unchanged; analysis and cleaning use in-memory working copies.
- Test identifiers and important fields instead of assuming each source table is reliable.
- Separate exact duplicates from repeated identifiers that have conflicting values.
- Treat payment amounts as **gross successful-payment amounts** unless reversals can be reliably linked to the original payment.
- Treat channel/targeting comparisons as descriptive evidence, not proof of causation.

## Scope and limitations
The supplied data only supports analysis over the dates actually present in the files. August 2026 is partial in this dataset, so comparisons of complete monthly totals should use January–July. A clean-looking table is not automatically the source of truth; unresolved conflicts should be documented rather than silently guessed away.

The setup cell below locates the project folder so the notebook does not depend on a personal Windows path.

In [2]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """Find the CredResolve project by locating its dataset/ folder.

    Works when Jupyter is launched from the project root, the notebooks folder,
    or the Desktop folder containing CredResolve.
    """
    start = (start or Path.cwd()).resolve()
    candidates = []
    for base in (start, *start.parents):
        candidates.append(base)
        candidates.append(base / "CredResolve")
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "dataset").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the CredResolve project. Start Jupyter from the "
        "CredResolve folder (or its notebooks folder) and ensure that the "
        "project contains a 'dataset' folder. Current working directory: "
        f"{Path.cwd().resolve()}"
    )


PROJECT_DIR = find_project_root()
DATASET_DIR = PROJECT_DIR / "dataset"
OUTPUTS_DIR = PROJECT_DIR / "outputs"
REPORTS_DIR = PROJECT_DIR / "reports"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_DIR)
print("Dataset folder exists:", DATASET_DIR.is_dir())
print("Outputs folder:", OUTPUTS_DIR)
print("Reports folder:", REPORTS_DIR)


Project root: C:\Users\GOPICHAND\Desktop\CredResolve
Dataset folder exists: True
Outputs folder: C:\Users\GOPICHAND\Desktop\CredResolve\outputs
Reports folder: C:\Users\GOPICHAND\Desktop\CredResolve\reports


## 1. Environment and project setup

Record the runtime versions and confirm the local project/data locations before analysis.

In [3]:
import sys
import pandas as pd
import numpy as np

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Python version: 3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]
Pandas version: 2.2.3
NumPy version: 2.2.0


In [4]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = find_project_root()
DATASET_DIR = PROJECT_DIR / "dataset"

print("Dataset folder exists:", DATASET_DIR.exists())
print("\nFiles found:")

for file in sorted(DATASET_DIR.iterdir()):
    print(file.name)

Dataset folder exists: True

Files found:
account_status_history.csv
accounts.csv
agent_sessions.csv
agents.csv
borrowers.csv
call_attempts.csv
call_dispositions.csv
calls.csv
campaigns.csv
complaints.csv
daily_targeting.csv
data_dictionary.csv
field_visits.csv
payments.csv
promises_to_pay.csv
README.md
sms_events.csv
vendor_telephony.csv
whatsapp_events.csv


## 2. Load and profile raw source tables

Load each CSV without changing the source files, then capture table sizes, columns, missingness, and duplicate-row counts.

In [5]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = find_project_root()
DATASET_DIR = PROJECT_DIR / "dataset"

# Load every CSV file without modifying the originals
data = {}

for file in sorted(DATASET_DIR.glob("*.csv")):
    try:
        data[file.stem] = pd.read_csv(file)
        print(f"{file.name}: {data[file.stem].shape}")
    except Exception as e:
        print(f"Error loading {file.name}: {e}")

print(f"\nTotal CSV tables loaded: {len(data)}")

account_status_history.csv: (60000, 8)
accounts.csv: (30000, 11)
agent_sessions.csv: (15000, 7)
agents.csv: (30000, 8)
borrowers.csv: (30600, 8)
call_attempts.csv: (120000, 9)
call_dispositions.csv: (35000, 8)
calls.csv: (91350, 11)
campaigns.csv: (120, 7)
complaints.csv: (8000, 9)
daily_targeting.csv: (45000, 7)
data_dictionary.csv: (143, 3)
field_visits.csv: (25000, 10)
payments.csv: (25500, 9)
promises_to_pay.csv: (18000, 9)
sms_events.csv: (45000, 8)
vendor_telephony.csv: (15, 6)
whatsapp_events.csv: (60600, 8)

Total CSV tables loaded: 18


In [6]:

for table_name, df in data.items():
    print(f"\n{'=' * 60}")
    print(f"TABLE: {table_name}")
    print(f"Rows and columns: {df.shape}")
    print("Columns:", df.columns.tolist())
    display(df.head(3))


TABLE: account_status_history
Rows and columns: (60000, 8)
Columns: ['history_id', 'account_id', 'borrower_id', 'event_at', 'status', 'changed_by', 'source', 'recorded_at']


,history_id,account_id,borrower_id,event_at,status,changed_by,source,recorded_at
0,HISTORY0000001,ACC0004322,BRW0005490,2026-07-03 04:28:38,WRITEOFF,AGT0000039,CALL,2026-07-03 00:41:54
1,HISTORY0000002,ACC0024644,BRW0005343,2026-07-18 02:03:23,DELINQUENT,AGT0000059,CALL,2026-07-18 22:42:02
2,HISTORY0000003,ACC0020233,BRW0003150,2026-04-15 09:17:47,CLOSED,AGT0000080,PAYMENT,2026-04-15 19:40:51



TABLE: accounts
Rows and columns: (30000, 11)
Columns: ['account_id', 'borrower_id', 'loan_type', 'principal_amount', 'outstanding_amount', 'dpd', 'risk_segment', 'status', 'opened_at', 'timezone', 'schema_version']


,account_id,borrower_id,loan_type,principal_amount,outstanding_amount,dpd,risk_segment,status,opened_at,timezone,schema_version
0,ACC0000001,BRW0010742,CONSUMER,603443.40,678074.03,15,MEDIUM,CLOSED,2025-11-11 04:37:00,Asia/Dubai,v1
1,ACC0000002,BRW0009382,BNPL,277190.14,464893.44,5,LOW,ACTIVE,2025-11-13 15:59:44,UTC,v3
2,ACC0000003,BRW0003966,CREDIT_CARD,628658.43,22565.37,60,NPA,WRITEOFF,2025-09-12 04:59:20,Asia/Dubai,v2



TABLE: agent_sessions
Rows and columns: (15000, 7)
Columns: ['session_id', 'agent_id', 'login_at', 'channel', 'device_id', 'timezone', 'logout_at']


...

,sms_event_id,account_id,borrower_id,event_at,message_id,event_type,template_code,provider_id
0,SMS_EVENT0000001,ACC0010751,BRW0009574,2026-04-24 20:57:06,SMS000022764,CLICKED,PAYMENT_LINK,VND0000010
1,SMS_EVENT0000002,ACC0005842,BRW0003619,2026-05-24 11:28:20,SMS000015050,DELIVERED,NOTICE,VND0000003
2,SMS_EVENT0000003,ACC0014253,BRW0003018,2026-03-05 20:23:20,SMS000028337,CLICKED,NOTICE,VND0000008



TABLE: vendor_telephony
Rows and columns: (15, 6)
Columns: ['vendor_id', 'vendor_name', 'vendor_account_id', 'timezone', 'status', 'schema_version']


,vendor_id,vendor_name,vendor_account_id,timezone,status,schema_version
0,VND0000001,Airtel,VAC342762,Asia/Kolkata,INACTIVE,v3
1,VND0000002,Exotel,VAC456766,UTC,INACTIVE,v3
2,VND0000003,Twilio,VAC074211,UTC,INACTIVE,v3



TABLE: whatsapp_events
Rows and columns: (60600, 8)
Columns: ['whatsapp_event_id', 'account_id', 'borrower_id', 'event_at', 'message_id', 'event_type', 'template_code', 'provider_id']


,whatsapp_event_id,account_id,borrower_id,event_at,message_id,event_type,template_code,provider_id
0,WHATSAPP_EVENT0000001,ACC0006134,BRW0007859,2026-02-27 17:49:38,MSG000008171,FAILED,FIELD_VISIT,VND0000008
1,WHATSAPP_EVENT0000002,ACC0027479,BRW0003963,2026-02-09 11:55:18,MSG000000667,FAILED,PTP_01,VND0000003
2,WHATSAPP_EVENT0000003,ACC0008356,BRW0008340,2026-07-08 05:16:13,MSG000023608,DELIVERED,PAYMENT_LINK,VND0000001


In [7]:

reports_dir = PROJECT_DIR / "reports"
reports_dir.mkdir(exist_ok=True)

profile = []

for table_name, df in data.items():
    profile.append({
        "table_name": table_name,
        "row_count": len(df),
        "column_count": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "total_missing_cells": int(df.isna().sum().sum()),
        "columns": ", ".join(df.columns.astype(str))
    })

profile_df = pd.DataFrame(profile)
profile_df.to_csv(reports_dir / "dataset_profile.csv", index=False)

display(profile_df)
print("Saved to:", reports_dir / "dataset_profile.csv")

,table_name,row_count,column_count,duplicate_rows,total_missing_cells,columns
0,account_status_history,60000,8,0,0,"history_id, account_id, borrower_id, event_at,..."
1,accounts,30000,11,0,455,"account_id, borrower_id, loan_type, principal_..."
2,agent_sessions,15000,7,0,0,"session_id, agent_id, login_at, channel, devic..."
3,agents,30000,8,0,0,"agent_id, employee_code, agent_name, vendor_id..."
4,borrowers,30600,8,600,1509,"borrower_id, name, phone, email, city, created..."
5,call_attempts,120000,9,0,2400,"attempt_id, account_id, borrower_id, event_at,..."
6,call_dispositions,35000,8,0,0,"disposition_id, account_id, borrower_id, event..."
7,calls,91350,11,1271,1827,"call_id, account_id, borrower_id, event_at, ag..."
8,campaigns,120,7,0,0,"campaign_id, campaign_name, channel, strategy_..."
9,complaints,8000,9,0,0,"complaint_id, account_id, borrower_id, event_a..."


Saved to: C:\Users\GOPICHAND\Desktop\CredResolve\reports\dataset_profile.csv


## 3. Key uniqueness, duplicates, and missing values

Check whether identifiers behave like reliable keys. Distinguish exact repeated records from the same ID appearing on conflicting records.

In [8]:

# 1. Check duplicate IDs in important tables

id_columns = {
    "accounts": "account_id",
    "borrowers": "borrower_id",
    "agents": "agent_id",
    "calls": "call_id",
    "payments": "payment_id",
    "promises_to_pay": "ptp_id",
    "whatsapp_events": "whatsapp_event_id",
    "sms_events": "sms_event_id"
}

for table_name, id_col in id_columns.items():
    df = data[table_name]

    duplicate_count = df[id_col].duplicated().sum()

    print(
        f"{table_name:20} | "
        f"Rows: {len(df):7,} | "
        f"Unique IDs: {df[id_col].nunique():7,} | "
        f"Repeated ID rows: {duplicate_count:6,}"
    )

accounts             | Rows:  30,000 | Unique IDs:  30,000 | Repeated ID rows:      0
borrowers            | Rows:  30,600 | Unique IDs:  11,015 | Repeated ID rows: 19,585
agents               | Rows:  30,000 | Unique IDs:   1,000 | Repeated ID rows: 29,000
calls                | Rows:  91,350 | Unique IDs:  90,000 | Repeated ID rows:  1,350
payments             | Rows:  25,500 | Unique IDs:  25,000 | Repeated ID rows:    500
promises_to_pay      | Rows:  18,000 | Unique IDs:  18,000 | Repeated ID rows:      0
whatsapp_events      | Rows:  60,600 | Unique IDs:  60,000 | Repeated ID rows:    600
sms_events           | Rows:  45,000 | Unique IDs:  45,000 | Repeated ID rows:      0


In [9]:

# 2. Show columns that contain missing values

for table_name, df in data.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    if not missing.empty:
        print(f"\n{table_name}")
        display(missing.rename("missing_count").to_frame())



accounts


,missing_count
borrower_id,455



borrowers


,missing_count
phone,614
email,895



call_attempts


...

,missing_count
agent_id,1827



field_visits


,missing_count
scheduled_at,250



payments


,missing_count
payment_reference,382


## 4. Payment-level forensics

Investigate repeated payment IDs, conflicting payment versions, transaction-reference reuse, status totals, and how deduplication changes the reported successful-payment amount.

In [10]:

# Inspect payment records with repeated payment IDs

payments = data["payments"]

duplicate_payments = payments[
    payments["payment_id"].duplicated(keep=False)
].sort_values("payment_id")

print("Rows involved in repeated payment IDs:",
      len(duplicate_payments))

display(duplicate_payments.head(20))


Rows involved in repeated payment IDs: 1000


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
55,PAYMENT0000056,ACC0019495,BRW0009716,2026-02-24 09:33:29,TXN0000008612,77982.73,PENDING,NETBANKING,VND0000007
25035,PAYMENT0000056,ACC0019495,BRW0009716,2026-02-24 09:33:29,TXN0000008612,77982.73,PENDING,NETBANKING,VND0000007
75,PAYMENT0000076,ACC0027928,BRW0010869,2026-02-05 17:14:30,TXN0000006417,43712.91,PENDING,UPI,VND0000014
25408,PAYMENT0000076,ACC0027928,BRW0010869,2026-02-05 17:14:30,TXN0000006417,43712.91,PENDING,UPI,VND0000014
107,PAYMENT0000108,ACC0017047,BRW0009175,2026-06-09 08:17:45,TXN0000041404,102830.42,SUCCESS,NACH,VND0000003
25481,PAYMENT0000108,ACC0017047,BRW0009175,2026-06-09 08:17:45,TXN0000041404,102830.42,SUCCESS,NACH,VND0000003
148,PAYMENT0000149,ACC0018444,BRW0005741,2026-07-25 03:04:14,TXN0000066565,96766.12,SUCCESS,CASH,VND0000013
25099,PAYMENT0000149,ACC0018444,BRW0005741,2026-07-25 03:04:14,TXN0000066565,96766.12,SUCCESS,CASH,VND0000013
198,PAYMENT0000199,ACC0000056,BRW0001872,2026-05-12 22:49:22,TXN0000069967,81814.00,SUCCESS,NETBANKING,VND0000012
25330,PAYMENT0000199,ACC0000056,BRW0001872,2026-05-12 22:49:22,TXN0000069967,81814.00,SUCCESS,NETBANKING,VND0000012


In [11]:

# Count distinct versions of each repeated payment ID

payment_id_check = (
    payments.groupby("payment_id")
    .agg(
        row_count=("payment_id", "size"),
        unique_accounts=("account_id", "nunique"),
        unique_amounts=("amount", "nunique"),
        unique_statuses=("payment_status", "nunique"),
        unique_references=("payment_reference", "nunique")
    )
)

repeated_payment_check = payment_id_check[
    payment_id_check["row_count"] > 1
].sort_values("row_count", ascending=False)

display(repeated_payment_check.head(20))

print("\nRepeated payment IDs:", len(repeated_payment_check))
print(
    "Repeated IDs with conflicting amounts:",
    (repeated_payment_check["unique_amounts"] > 1).sum()
)
print(
    "Repeated IDs with conflicting statuses:",
    (repeated_payment_check["unique_statuses"] > 1).sum()
)

,row_count,unique_accounts,unique_amounts,unique_statuses,unique_references
payment_id,,,,,
PAYMENT0024852,2,1,1,1,1
PAYMENT0000056,2,1,1,1,1
PAYMENT0000076,2,1,1,1,1
PAYMENT0000108,2,1,1,1,1
PAYMENT0024310,2,1,1,1,1
PAYMENT0024235,2,1,1,1,1
PAYMENT0024177,2,1,1,1,1
PAYMENT0024144,2,1,1,1,1
PAYMENT0024143,2,1,1,1,1



Repeated payment IDs: 500
Repeated IDs with conflicting amounts: 0
Repeated IDs with conflicting statuses: 0


In [12]:

# Check whether repeated payment IDs have identical full records

repeated_ids = payments.loc[
    payments["payment_id"].duplicated(keep=False),
    "payment_id"
].unique()

repeated_records = payments[
    payments["payment_id"].isin(repeated_ids)
]

rows_within_repeated_ids = len(repeated_records)

unique_full_records = len(repeated_records.drop_duplicates())

print("Rows belonging to repeated payment IDs:", rows_within_repeated_ids)
print("Unique full records among those rows:", unique_full_records)
print(
    "Exact duplicate copies among these rows:",
    rows_within_repeated_ids - unique_full_records
)

# Check whether any repeated ID has more than one distinct full record
versions_per_id = repeated_records.groupby("payment_id").apply(
    lambda group: len(group.drop_duplicates()),
    include_groups=False
)

print(
    "Repeated IDs with differing full records:",
    (versions_per_id > 1).sum()
)


Rows belonging to repeated payment IDs: 1000
Unique full records among those rows: 514
Exact duplicate copies among these rows: 486
Repeated IDs with differing full records: 14


In [13]:

# Payment status summary before deduplication

payment_summary = (
    payments.groupby("payment_status")
    .agg(
        transaction_rows=("payment_id", "size"),
        unique_payment_ids=("payment_id", "nunique"),
        total_amount=("amount", "sum")
    )
    .sort_values("transaction_rows", ascending=False)
)

display(payment_summary)

# Show successful-payment totals before and after removing exact duplicate rows
successful_before = payments.loc[
    payments["payment_status"].eq("SUCCESS"), "amount"
].sum()

payments_exact_dedup = payments.drop_duplicates()

successful_after = payments_exact_dedup.loc[
    payments_exact_dedup["payment_status"].eq("SUCCESS"), "amount"
].sum()

print(f"SUCCESS amount before exact deduplication: ₹{successful_before:,.2f}")
print(f"SUCCESS amount after exact deduplication:  ₹{successful_after:,.2f}")
print(f"Difference: ₹{successful_before - successful_after:,.2f}")

,transaction_rows,unique_payment_ids,total_amount
payment_status,,,
SUCCESS,17880,17534,1.341486e+09
FAILED,3744,3677,2.835063e+08
PENDING,2592,2535,1.948680e+08
REVERSED,1284,1254,9.739842e+07


SUCCESS amount before exact deduplication: ₹1,341,485,926.33
SUCCESS amount after exact deduplication:  ₹1,316,474,464.14
Difference: ₹25,011,462.19


In [14]:

# Identify payment IDs with more than one distinct full record

payment_versions = (
    payments.groupby("payment_id")
    .apply(lambda group: len(group.drop_duplicates()))
)

conflicting_ids = payment_versions[payment_versions > 1].index

conflicting_payments = payments[
    payments["payment_id"].isin(conflicting_ids)
].sort_values(["payment_id", "event_at"])

print("Conflicting payment IDs:", len(conflicting_ids))

display(conflicting_payments)


Conflicting payment IDs: 14


C:\Users\GOPICHAND\AppData\Local\Temp\ipykernel_32168\2277577108.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: len(group.drop_duplicates()))


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
1389,PAYMENT0001390,ACC0001624,BRW0004200,2026-05-14 07:52:59,TXN0000046535,85205.16,SUCCESS,CASH,VND0000015
25043,PAYMENT0001390,ACC0001624,BRW0004200,2026-05-14 07:52:59,NaN,85205.16,SUCCESS,CASH,VND0000015
2097,PAYMENT0002098,ACC0023839,BRW0008653,2026-07-01 09:19:16,TXN0000001302,5570.92,SUCCESS,NETBANKING,VND0000001
25055,PAYMENT0002098,ACC0023839,BRW0008653,2026-07-01 09:19:16,NaN,5570.92,SUCCESS,NETBANKING,VND0000001
2302,PAYMENT0002303,ACC0021362,BRW0001779,2026-06-15 03:09:47,TXN0000067311,59429.86,SUCCESS,CARD,VND0000011
25057,PAYMENT0002303,ACC0021362,BRW0001779,2026-06-15 03:09:47,NaN,59429.86,SUCCESS,CARD,VND0000011
5190,PAYMENT0005191,ACC0001901,BRW0001234,2026-06-28 13:50:03,NaN,107235.47,SUCCESS,NACH,VND0000002
25437,PAYMENT0005191,ACC0001901,BRW0001234,2026-06-28 13:50:03,TXN0000041890,107235.47,SUCCESS,NACH,VND0000002
7581,PAYMENT0007582,ACC0017962,BRW0008041,2026-03-26 12:49:42,NaN,119714.20,SUCCESS,NETBANKING,VND0000014
25385,PAYMENT0007582,ACC0017962,BRW0008041,2026-03-26 12:49:42,TXN0000013106,119714.20,SUCCESS,NETBANKING,VND0000014


In [15]:

# Create a separate working copy
payments_clean = payments.drop_duplicates().copy()

print("Original payment rows:", len(payments))
print("Rows after exact deduplication:", len(payments_clean))
print("Exact duplicate rows removed:", len(payments) - len(payments_clean))

# Summarize payment statuses after exact deduplication
payment_status_summary = (
    payments_clean.groupby("payment_status")
    .agg(
        transaction_rows=("payment_id", "size"),
        unique_payment_ids=("payment_id", "nunique"),
        total_amount=("amount", "sum")
    )
)

display(payment_status_summary)


Original payment rows: 25500
Rows after exact deduplication: 25014
Exact duplicate rows removed: 486


,transaction_rows,unique_payment_ids,total_amount
payment_status,,,
FAILED,3678,3677,2.784453e+08
PENDING,2536,2535,1.903653e+08
REVERSED,1255,1254,9.467448e+07
SUCCESS,17545,17534,1.316474e+09


In [16]:

# Investigate the 14 payment IDs with different full records

conflict_check = (
    payments[payments["payment_id"].isin(conflicting_ids)]
    .groupby("payment_id")
    .agg(
        row_count=("payment_id", "size"),
        account_count=("account_id", "nunique"),
        borrower_count=("borrower_id", "nunique"),
        timestamp_count=("event_at", "nunique"),
        amount_count=("amount", "nunique"),
        status_count=("payment_status", "nunique"),
        method_count=("payment_method", "nunique"),
        provider_count=("provider_id", "nunique"),
        reference_count=("payment_reference", "nunique"),
        missing_references=("payment_reference", lambda x: x.isna().sum())
    )
    .reset_index()
)

display(conflict_check)

,payment_id,row_count,account_count,borrower_count,timestamp_count,amount_count,status_count,method_count,provider_count,reference_count,missing_references
0,PAYMENT0001390,2,1,1,1,1,1,1,1,1,1
1,PAYMENT0002098,2,1,1,1,1,1,1,1,1,1
2,PAYMENT0002303,2,1,1,1,1,1,1,1,1,1
3,PAYMENT0005191,2,1,1,1,1,1,1,1,1,1
4,PAYMENT0007582,2,1,1,1,1,1,1,1,1,1
5,PAYMENT0007751,2,1,1,1,1,1,1,1,1,1
6,PAYMENT0008752,2,1,1,1,1,1,1,1,1,1
7,PAYMENT0010558,2,1,1,1,1,1,1,1,1,1
8,PAYMENT0016862,2,1,1,1,1,1,1,1,1,1
9,PAYMENT0018134,2,1,1,1,1,1,1,1,1,1


In [17]:

# Start with exact duplicate removal
payments_clean = payments.drop_duplicates().copy()

# Fields that should agree for records representing the same payment
key_fields = [
    "account_id",
    "borrower_id",
    "event_at",
    "amount",
    "payment_status",
    "payment_method",
    "provider_id"
]

# Check whether repeated payment IDs conflict on these key fields
key_conflicts = (
    payments_clean.groupby("payment_id")[key_fields]
    .nunique(dropna=False)
    .gt(1)
    .any(axis=1)
)

conflicting_key_ids = key_conflicts[key_conflicts].index

# Keep repeated IDs with conflicting key details separate for investigation
safe_records = payments_clean[
    ~payments_clean["payment_id"].isin(conflicting_key_ids)
].copy()

review_records = payments_clean[
    payments_clean["payment_id"].isin(conflicting_key_ids)
].copy()

# Consolidate only the repeated IDs whose key details agree
# Preserve the available transaction reference
def combine_payment(group):
    row = group.iloc[0].copy()

    for col in group.columns:
        non_missing = group[col].dropna().unique()

        if col == "payment_reference" and len(non_missing) >= 1:
            row[col] = non_missing[0]

    return row

payments_consolidated = (
    safe_records.groupby("payment_id", sort=False, group_keys=False)
    .apply(combine_payment, include_groups=False)
    .reset_index(drop=True)
)

# Add records with conflicting key details for manual review
payments_clean_review = pd.concat(
    [payments_consolidated, review_records],
    ignore_index=True
)

print("Rows after exact deduplication:", len(payments_clean))
print("Payment IDs with conflicting key details:", len(conflicting_key_ids))
print("Rows in consolidated table:", len(payments_consolidated))
print("Rows requiring manual review:", len(review_records))

display(payments_clean_review.head())

Rows after exact deduplication: 25014
Payment IDs with conflicting key details: 0
Rows in consolidated table: 25000
Rows requiring manual review: 0


,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id,payment_id
0,ACC0015539,BRW0011363,2026-02-27 01:28:12,TXN0000007457,22433.23,FAILED,CARD,VND0000001,NaN
1,ACC0004445,BRW0011306,2026-07-23 20:25:20,TXN0000030016,23295.11,FAILED,UPI,VND0000004,NaN
2,ACC0003853,BRW0002549,2026-01-11 21:27:46,TXN0000010466,118814.97,FAILED,NACH,VND0000013,NaN
3,ACC0017531,BRW0006618,2026-06-16 02:35:21,TXN0000031781,145082.17,REVERSED,CASH,VND0000013,NaN
4,ACC0001770,BRW0011492,2026-03-03 06:08:23,TXN0000059257,115148.46,SUCCESS,NETBANKING,VND0000013,NaN


In [18]:

# Rebuild the consolidated payment table, preserving payment_id

def combine_payment(group):
    row = group.iloc[0].copy()

    # Retain the payment ID
    row["payment_id"] = group.name

    # Retain a transaction reference if one exists
    references = group["payment_reference"].dropna().unique()
    if len(references) > 0:
        row["payment_reference"] = references[0]

    return row

payments_consolidated = (
    safe_records.groupby("payment_id", sort=False)
    .apply(combine_payment)
    .reset_index(drop=True)
)

print("Rows:", len(payments_consolidated))
print("Unique payment IDs:", payments_consolidated["payment_id"].nunique())
print("Missing payment IDs:", payments_consolidated["payment_id"].isna().sum())

display(payments_consolidated.head())


Rows: 25000
Unique payment IDs: 25000
Missing payment IDs: 0


C:\Users\GOPICHAND\AppData\Local\Temp\ipykernel_32168\3589362411.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(combine_payment)


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
0,PAYMENT0000001,ACC0015539,BRW0011363,2026-02-27 01:28:12,TXN0000007457,22433.23,FAILED,CARD,VND0000001
1,PAYMENT0000002,ACC0004445,BRW0011306,2026-07-23 20:25:20,TXN0000030016,23295.11,FAILED,UPI,VND0000004
2,PAYMENT0000003,ACC0003853,BRW0002549,2026-01-11 21:27:46,TXN0000010466,118814.97,FAILED,NACH,VND0000013
3,PAYMENT0000004,ACC0017531,BRW0006618,2026-06-16 02:35:21,TXN0000031781,145082.17,REVERSED,CASH,VND0000013
4,PAYMENT0000005,ACC0001770,BRW0011492,2026-03-03 06:08:23,TXN0000059257,115148.46,SUCCESS,NETBANKING,VND0000013


In [19]:

# Parse payment timestamps
payments_consolidated["event_at"] = pd.to_datetime(
    payments_consolidated["event_at"],
    errors="coerce"
)

# Summarize payment outcomes
monthly_payments = (
    payments_consolidated
    .assign(month=payments_consolidated["event_at"].dt.to_period("M"))
    .groupby(["month", "payment_status"])
    .agg(
        transaction_count=("payment_id", "nunique"),
        amount=("amount", "sum")
    )
    .reset_index()
)

display(monthly_payments)


,month,payment_status,transaction_count,amount
0,2026-01,FAILED,515,3.799883e+07
1,2026-01,PENDING,375,2.855892e+07
2,2026-01,REVERSED,174,1.163123e+07
3,2026-01,SUCCESS,2464,1.872291e+08
4,2026-02,FAILED,451,3.494205e+07
5,2026-02,PENDING,312,2.294651e+07
6,2026-02,REVERSED,140,1.147768e+07
7,2026-02,SUCCESS,2268,1.701425e+08
8,2026-03,FAILED,533,4.093328e+07
9,2026-03,PENDING,334,2.487168e+07


In [20]:

# Check payment status distribution after consolidation

status_summary = (
    payments_consolidated
    .groupby("payment_status")
    .agg(
        payment_count=("payment_id", "nunique"),
        total_amount=("amount", "sum")
    )
    .reset_index()
)

status_summary["total_amount_crore"] = (
    status_summary["total_amount"] / 10_000_000
)

display(status_summary)


,payment_status,payment_count,total_amount,total_amount_crore
0,FAILED,3677,2.784186e+08,27.841857
1,PENDING,2535,1.902232e+08,19.022318
2,REVERSED,1254,9.466657e+07,9.466657
3,SUCCESS,17534,1.315584e+09,131.558396


In [21]:

# Check for transaction references linked to multiple payment IDs

reference_check = (
    payments_consolidated
    .dropna(subset=["payment_reference"])
    .groupby("payment_reference")
    .agg(
        payment_ids=("payment_id", "nunique"),
        accounts=("account_id", "nunique"),
        amounts=("amount", "nunique")
    )
    .reset_index()
)

reference_conflicts = reference_check[
    (reference_check["payment_ids"] > 1) |
    (reference_check["accounts"] > 1) |
    (reference_check["amounts"] > 1)
]

print("Transaction references reused across records:", len(reference_conflicts))
display(reference_conflicts.head(20))

Transaction references reused across records: 3407


,payment_reference,payment_ids,accounts,amounts
5,TXN0000000027,2,2,2
8,TXN0000000032,3,3,3
28,TXN0000000113,3,3,3
35,TXN0000000132,2,2,2
36,TXN0000000134,2,2,2
39,TXN0000000154,2,2,2
50,TXN0000000186,2,2,2
53,TXN0000000196,2,2,2
58,TXN0000000212,2,2,2
71,TXN0000000271,2,2,2


In [22]:

# Summarize transaction-reference reuse

reference_summary = (
    payments_consolidated
    .dropna(subset=["payment_reference"])
    .groupby("payment_reference")
    .agg(
        payment_ids=("payment_id", "nunique"),
        accounts=("account_id", "nunique"),
        amounts=("amount", "nunique")
    )
    .reset_index()
)

total_references = len(reference_summary)
reused_references = (reference_summary["payment_ids"] > 1).sum()

print("Unique non-missing transaction references:", total_references)
print("References used by multiple payment IDs:", reused_references)
print(
    "Percentage of references reused:",
    round(reused_references / total_references * 100, 2),
    "%"
)

print("\nPayment IDs linked to reused references:")
print(
    reference_summary.loc[
        reference_summary["payment_ids"] > 1, "payment_ids"
    ].sum()
)

print("\nLargest reference-reuse groups:")
display(
    reference_summary
    .sort_values("payment_ids", ascending=False)
    .head(15)
)


Unique non-missing transaction references: 20821
References used by multiple payment IDs: 3407
Percentage of references reused: 16.36 %

Payment IDs linked to reused references:
7218

Largest reference-reuse groups:


,payment_reference,payment_ids,accounts,amounts
13081,TXN0000044312,5,5,5
13088,TXN0000044341,4,4,4
20078,TXN0000067557,4,4,4
515,TXN0000001778,4,4,4
585,TXN0000002005,4,4,4
18924,TXN0000063848,4,4,4
14050,TXN0000047758,4,4,4
15513,TXN0000052559,4,4,4
15649,TXN0000052986,4,4,4
19524,TXN0000065723,4,4,4


## 5. Account portfolio and monthly performance

Inspect account status and portfolio composition, then calculate monthly gross successful-payment trends. Avoid treating this amount alone as net recovery.

In [23]:

# Inspect account fields
accounts = data["accounts"]

print("Accounts columns:")
print(accounts.columns.tolist())

display(accounts.head())

print("\nData dictionary entries related to accounts and payments:")

dictionary = data["data_dictionary"]

relevant_entries = dictionary[
    dictionary.astype(str)
    .apply(
        lambda row: row.str.contains(
            "outstanding|balance|payment|recovery|principal|amount|DPD",
            case=False,
            na=False
        ).any(),
        axis=1
    )
]

display(relevant_entries)

Accounts columns:
['account_id', 'borrower_id', 'loan_type', 'principal_amount', 'outstanding_amount', 'dpd', 'risk_segment', 'status', 'opened_at', 'timezone', 'schema_version']


,account_id,borrower_id,loan_type,principal_amount,outstanding_amount,dpd,risk_segment,status,opened_at,timezone,schema_version
0,ACC0000001,BRW0010742,CONSUMER,603443.40,678074.03,15,MEDIUM,CLOSED,2025-11-11 04:37:00,Asia/Dubai,v1
1,ACC0000002,BRW0009382,BNPL,277190.14,464893.44,5,LOW,ACTIVE,2025-11-13 15:59:44,UTC,v3
2,ACC0000003,BRW0003966,CREDIT_CARD,628658.43,22565.37,60,NPA,WRITEOFF,2025-09-12 04:59:20,Asia/Dubai,v2
3,ACC0000004,BRW0001993,PERSONAL,76435.07,508427.73,30,LOW,CLOSED,2025-05-25 19:29:38,UTC,v2
4,ACC0000005,BRW0009976,CONSUMER,646797.08,563858.01,180,LOW,WRITEOFF,2025-09-07 14:59:13,Asia/Dubai,v2



Data dictionary entries related to accounts and payments:


,dataset,column,dtype
11,accounts,principal_amount,float64
12,accounts,outstanding_amount,float64
13,accounts,dpd,int64
107,promises_to_pay,promised_amount,float64
111,payments,payment_id,object
112,payments,account_id,object
113,payments,borrower_id,object
114,payments,event_at,datetime64[ns]
115,payments,payment_reference,object
116,payments,amount,float64


In [24]:

accounts = data["accounts"]

print("Total accounts:", len(accounts))
print("\nAccount status:")
display(accounts["status"].value_counts(dropna=False))

print("\nDays past due (DPD) distribution:")
display(accounts["dpd"].describe())

print("\nOutstanding amount summary:")
display(accounts["outstanding_amount"].describe())

print("\nAccounts by risk segment:")
display(accounts["risk_segment"].value_counts(dropna=False))

print("\nAccounts by loan type:")
display(accounts["loan_type"].value_counts(dropna=False))

print("\nMissing values in key fields:")
display(
    accounts[
        [
            "account_id",
            "outstanding_amount",
            "dpd",
            "risk_segment",
            "status"
        ]
    ].isna().sum()
)


Total accounts: 30000

Account status:


status
ACTIVE      7539
CLOSED      7496
PAID        7486
WRITEOFF    7479
Name: count, dtype: int64


Days past due (DPD) distribution:


count    30000.000000
mean        56.506433
std         53.970460
min          0.000000
25%          5.000000
50%         45.000000
75%         90.000000
max        180.000000
Name: dpd, dtype: float64


Outstanding amount summary:


...

risk_segment
HIGH      7552
MEDIUM    7533
LOW       7513
NPA       7402
Name: count, dtype: int64


Accounts by loan type:


loan_type
CREDIT_CARD    6080
AUTO           6079
PERSONAL       5983
CONSUMER       5930
BNPL           5928
Name: count, dtype: int64


Missing values in key fields:


account_id            0
outstanding_amount    0
dpd                   0
risk_segment          0
status                0
dtype: int64

In [25]:

accounts = data["accounts"]

# Portfolio summary by account status
status_summary = (
    accounts.groupby("status")
    .agg(
        account_count=("account_id", "nunique"),
        outstanding_amount=("outstanding_amount", "sum"),
        average_dpd=("dpd", "mean")
    )
    .reset_index()
)

status_summary["outstanding_crore"] = (
    status_summary["outstanding_amount"] / 10_000_000
)

print("Portfolio by account status:")
display(status_summary)

# Portfolio summary by risk segment
risk_summary = (
    accounts.groupby("risk_segment")
    .agg(
        account_count=("account_id", "nunique"),
        outstanding_amount=("outstanding_amount", "sum"),
        average_dpd=("dpd", "mean")
    )
    .reset_index()
)

risk_summary["outstanding_crore"] = (
    risk_summary["outstanding_amount"] / 10_000_000
)

print("\nPortfolio by risk segment:")
display(risk_summary)

# Overdue-account distribution
print("\nAccounts by DPD bucket:")

accounts["dpd_bucket"] = pd.cut(
    accounts["dpd"],
    bins=[-1, 0, 30, 60, 90, 180],
    labels=["0 days", "1–30 days", "31–60 days", "61–90 days", "91–180 days"]
)

dpd_summary = (
    accounts.groupby("dpd_bucket", observed=False)
    .agg(
        account_count=("account_id", "nunique"),
        outstanding_amount=("outstanding_amount", "sum")
    )
    .reset_index()
)

dpd_summary["outstanding_crore"] = (
    dpd_summary["outstanding_amount"] / 10_000_000
)

display(dpd_summary)

Portfolio by account status:


,status,account_count,outstanding_amount,average_dpd,outstanding_crore
0,ACTIVE,7539,2.637397e+09,56.257992,263.739684
1,CLOSED,7496,2.636143e+09,55.913154,263.614325
2,PAID,7486,2.617540e+09,57.263024,261.753954
3,WRITEOFF,7479,2.597956e+09,56.594197,259.795572



Portfolio by risk segment:


,risk_segment,account_count,outstanding_amount,average_dpd,outstanding_crore
0,HIGH,7552,2.646183e+09,56.447961,264.618259
1,LOW,7513,2.633311e+09,57.972048,263.331110
2,MEDIUM,7533,2.628179e+09,54.763972,262.817946
3,NPA,7402,2.581362e+09,56.851797,258.136220



Accounts by DPD bucket:


,dpd_bucket,account_count,outstanding_amount,outstanding_crore
0,0 days,2685,9.482186e+08,94.821863
1,1–30 days,10880,3.834101e+09,383.410052
2,31–60 days,5514,1.914490e+09,191.449030
3,61–90 days,5468,1.895868e+09,189.586840
4,91–180 days,5453,1.896357e+09,189.635750


In [26]:
# Investigate successful payments by account status

payments = payments_consolidated.copy()
accounts = data["accounts"].copy()

# Convert payment timestamps and amounts
payments["event_at"] = pd.to_datetime(
    payments["event_at"], errors="coerce", utc=True
)
payments["amount"] = pd.to_numeric(
    payments["amount"], errors="coerce"
)

# Keep only successful payments for this initial analysis
successful_payments = payments[
    payments["payment_status"].eq("SUCCESS")
].copy()

# Summarize successful payments for each account
account_payments = (
    successful_payments.groupby("account_id")
    .agg(
        successful_payment_count=("payment_id", "nunique"),
        successful_payment_amount=("amount", "sum"),
        first_successful_payment=("event_at", "min"),
        last_successful_payment=("event_at", "max")
    )
    .reset_index()
)

# Join payment summaries to the account table
account_payment_analysis = accounts.merge(
    account_payments,
    on="account_id",
    how="left",
    validate="one_to_one"
)

# Replace missing payment summaries with zero
account_payment_analysis["successful_payment_count"] = (
    account_payment_analysis["successful_payment_count"].fillna(0)
)
account_payment_analysis["successful_payment_amount"] = (
    account_payment_analysis["successful_payment_amount"].fillna(0)
)

# Summarize by account status
status_payment_summary = (
    account_payment_analysis.groupby("status")
    .agg(
        account_count=("account_id", "nunique"),
        accounts_with_successful_payments=(
            "successful_payment_count",
            lambda x: (x > 0).sum()
        ),
        successful_payment_count=(
            "successful_payment_count", "sum"
        ),
        successful_payment_amount=(
            "successful_payment_amount", "sum"
        )
    )
    .reset_index()
)

status_payment_summary["accounts_with_payments_pct"] = (
    status_payment_summary["accounts_with_successful_payments"]
    / status_payment_summary["account_count"] * 100
)

status_payment_summary["successful_payment_crore"] = (
    status_payment_summary["successful_payment_amount"]
    / 10_000_000
)

display(status_payment_summary.round(2))

,status,account_count,accounts_with_successful_payments,successful_payment_count,successful_payment_amount,accounts_with_payments_pct,successful_payment_crore
0,ACTIVE,7539,3315,4404.0,3.329433e+08,43.97,33.29
1,CLOSED,7496,3325,4381.0,3.256834e+08,44.36,32.57
2,PAID,7486,3297,4379.0,3.287247e+08,44.04,32.87
3,WRITEOFF,7479,3347,4370.0,3.282327e+08,44.75,32.82


In [27]:


payments = payments_consolidated.copy()

payments["event_at"] = pd.to_datetime(
    payments["event_at"],
    errors="coerce",
    utc=True
)

payments["amount"] = pd.to_numeric(
    payments["amount"],
    errors="coerce"
)

payments["month"] = (
    payments["event_at"]
    .dt.to_period("M")
    .astype(str)
)

monthly_status = (
    payments.groupby(["month", "payment_status"])
    .agg(
        transaction_count=("payment_id", "nunique"),
        total_amount=("amount", "sum")
    )
    .reset_index()
)

monthly_status["amount_crore"] = (
    monthly_status["total_amount"] / 10_000_000
)

display(
    monthly_status[
        [
            "month",
            "payment_status",
            "transaction_count",
            "amount_crore"
        ]
    ].round(2)
)

C:\Users\GOPICHAND\AppData\Local\Temp\ipykernel_32168\1025964883.py:16: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


,month,payment_status,transaction_count,amount_crore
0,2026-01,FAILED,515,3.80
1,2026-01,PENDING,375,2.86
2,2026-01,REVERSED,174,1.16
3,2026-01,SUCCESS,2464,18.72
4,2026-02,FAILED,451,3.49
5,2026-02,PENDING,312,2.29
6,2026-02,REVERSED,140,1.15
7,2026-02,SUCCESS,2268,17.01
8,2026-03,FAILED,533,4.09
9,2026-03,PENDING,334,2.49


In [28]:
# Calculate monthly successful payment trends

successful_monthly = (
    payments[payments["payment_status"] == "SUCCESS"]
    .groupby("month")
    .agg(
        successful_transactions=("payment_id", "nunique"),
        successful_amount=("amount", "sum")
    )
    .reset_index()
    .sort_values("month")
)

# Calculate month-on-month percentage changes
successful_monthly["amount_mom_change_pct"] = (
    successful_monthly["successful_amount"]
    .pct_change() * 100
)

successful_monthly["transaction_mom_change_pct"] = (
    successful_monthly["successful_transactions"]
    .pct_change() * 100
)

successful_monthly["amount_crore"] = (
    successful_monthly["successful_amount"] / 10_000_000
)

display(
    successful_monthly[
        [
            "month",
            "successful_transactions",
            "amount_crore",
            "amount_mom_change_pct",
            "transaction_mom_change_pct"
        ]
    ].round(2)
)

,month,successful_transactions,amount_crore,amount_mom_change_pct,transaction_mom_change_pct
0,2026-01,2464,18.72,NaN,NaN
1,2026-02,2268,17.01,-9.13,-7.95
2,2026-03,2524,18.89,11.03,11.29
3,2026-04,2406,17.51,-7.29,-4.68
4,2026-05,2449,18.43,5.20,1.79
5,2026-06,2366,17.56,-4.72,-3.39
6,2026-07,2441,18.72,6.65,3.17
7,2026-08,616,4.71,-74.84,-74.76


## 6. Targeting and campaign data checks

Check targeting dates, campaign mappings, and strategy changes. These comparisons can reveal mix changes, but observational associations do not establish causation.

In [29]:
# Inspect account history and targeting data

account_history = data["account_status_history"]
targeting = data["daily_targeting"]

print("ACCOUNT STATUS HISTORY")
print("Shape:", account_history.shape)
print("Columns:", account_history.columns.tolist())
display(account_history.head())

print("\nDAILY TARGETING")
print("Shape:", targeting.shape)
print("Columns:", targeting.columns.tolist())
display(targeting.head())

ACCOUNT STATUS HISTORY
Shape: (60000, 8)
Columns: ['history_id', 'account_id', 'borrower_id', 'event_at', 'status', 'changed_by', 'source', 'recorded_at']


,history_id,account_id,borrower_id,event_at,status,changed_by,source,recorded_at
0,HISTORY0000001,ACC0004322,BRW0005490,2026-07-03 04:28:38,WRITEOFF,AGT0000039,CALL,2026-07-03 00:41:54
1,HISTORY0000002,ACC0024644,BRW0005343,2026-07-18 02:03:23,DELINQUENT,AGT0000059,CALL,2026-07-18 22:42:02
2,HISTORY0000003,ACC0020233,BRW0003150,2026-04-15 09:17:47,CLOSED,AGT0000080,PAYMENT,2026-04-15 19:40:51
3,HISTORY0000004,ACC0004384,BRW0008483,2026-01-16 10:52:04,WRITEOFF,SYSTEM,CORE,2026-01-16 23:05:28
4,HISTORY0000005,ACC0022967,BRW0009477,2026-05-22 04:13:04,PTP,AGT0000073,CORE,2026-05-22 07:08:28



DAILY TARGETING
Shape: (45000, 7)
Columns: ['target_id', 'account_id', 'campaign_id', 'target_date', 'priority', 'recommended_channel', 'status']


,target_id,account_id,campaign_id,target_date,priority,recommended_channel,status
0,TGT0000001,ACC0028555,CMP0000103,2026-04-22,4,FIELD,QUEUED
1,TGT0000002,ACC0007194,CMP0000100,2026-08-06,10,SMS,EXPIRED
2,TGT0000003,ACC0029550,CMP0000074,2026-05-20,5,SMS,CONTACTED
3,TGT0000004,ACC0012329,CMP0000046,2026-07-30,10,WHATSAPP,QUEUED
4,TGT0000005,ACC0018387,CMP0000118,2026-07-10,7,WHATSAPP,EXPIRED


In [30]:
# Step 1: Check dates, duplicates, and missing values

history = data["account_status_history"].copy()
targeting = data["daily_targeting"].copy()

# Convert dates
history["event_at"] = pd.to_datetime(
    history["event_at"], errors="coerce"
)
history["recorded_at"] = pd.to_datetime(
    history["recorded_at"], errors="coerce"
)
targeting["target_date"] = pd.to_datetime(
    targeting["target_date"], errors="coerce"
)

print("=== ACCOUNT STATUS HISTORY ===")
print("Date range:", history["event_at"].min(), "to", history["event_at"].max())
print("Unique history IDs:", history["history_id"].nunique())
print("Unique accounts:", history["account_id"].nunique())
print("Duplicate history IDs:", history["history_id"].duplicated().sum())
print("\nMissing values:")
display(history.isna().sum().to_frame("missing_count"))

print("\nStatus event counts:")
display(history["status"].value_counts(dropna=False).to_frame("event_count"))

print("\n=== DAILY TARGETING ===")
print("Date range:", targeting["target_date"].min(), "to", targeting["target_date"].max())
print("Unique target IDs:", targeting["target_id"].nunique())
print("Unique accounts targeted:", targeting["account_id"].nunique())
print("Duplicate target IDs:", targeting["target_id"].duplicated().sum())
print("\nMissing values:")
display(targeting.isna().sum().to_frame("missing_count"))

print("\nTargeting status counts:")
display(targeting["status"].value_counts(dropna=False).to_frame("target_count"))

print("\nRecommended channel counts:")
display(targeting["recommended_channel"].value_counts(dropna=False).to_frame("target_count"))

=== ACCOUNT STATUS HISTORY ===
Date range: 2026-01-01 00:01:08 to 2026-08-08 23:50:45
Unique history IDs: 60000
Unique accounts: 25999
Duplicate history IDs: 0

Missing values:


,missing_count
history_id,0
account_id,0
borrower_id,0
event_at,0
status,0
changed_by,0
source,0
recorded_at,0



Status event counts:


,event_count
status,
PAID,8650
CLOSED,8614
DELINQUENT,8612
NPA,8612
WRITEOFF,8583
ACTIVE,8518
PTP,8411



=== DAILY TARGETING ===
Date range: 2026-01-01 00:00:00 to 2026-08-08 00:00:00
Unique target IDs: 45000
Unique accounts targeted: 23344
Duplicate target IDs: 0

Missing values:


,missing_count
target_id,0
account_id,0
campaign_id,0
target_date,0
priority,0
recommended_channel,0
status,0



Targeting status counts:


,target_count
status,
EXPIRED,11371
CONTACTED,11254
QUEUED,11202
SKIPPED,11173



Recommended channel counts:


,target_count
recommended_channel,
FIELD,11365
SMS,11221
WHATSAPP,11212
VOICE,11202


In [31]:
# Monthly targeting analysis

targeting = data["daily_targeting"].copy()

targeting["target_date"] = pd.to_datetime(
    targeting["target_date"],
    errors="coerce"
)

targeting["month"] = targeting["target_date"].dt.strftime("%Y-%m")

# Count targets by month and targeting status
monthly_target_status = (
    targeting.groupby(["month", "status"])
    .agg(target_count=("target_id", "nunique"))
    .reset_index()
)

print("MONTHLY TARGETING STATUS")
display(
    monthly_target_status.pivot(
        index="month",
        columns="status",
        values="target_count"
    ).fillna(0).astype(int)
)

# Count targets by month and recommended channel
monthly_target_channel = (
    targeting.groupby(["month", "recommended_channel"])
    .agg(target_count=("target_id", "nunique"))
    .reset_index()
)

print("\nMONTHLY RECOMMENDED CHANNEL")
display(
    monthly_target_channel.pivot(
        index="month",
        columns="recommended_channel",
        values="target_count"
    ).fillna(0).astype(int)
)

MONTHLY TARGETING STATUS


status,CONTACTED,EXPIRED,QUEUED,SKIPPED
month,,,,
2026-01,1562,1615,1553,1639
2026-02,1409,1473,1454,1373
2026-03,1564,1590,1609,1527
2026-04,1577,1560,1515,1553
2026-05,1617,1597,1605,1623
2026-06,1547,1594,1477,1536
2026-07,1571,1537,1589,1533
2026-08,407,405,400,389



MONTHLY RECOMMENDED CHANNEL


recommended_channel,FIELD,SMS,VOICE,WHATSAPP
month,,,,
2026-01,1602,1608,1543,1616
2026-02,1471,1379,1429,1430
2026-03,1562,1616,1524,1588
2026-04,1632,1527,1512,1534
2026-05,1575,1618,1698,1551
2026-06,1571,1543,1533,1507
2026-07,1568,1535,1567,1560
2026-08,384,395,396,426


In [32]:
# Inspect campaign information

campaigns = data["campaigns"].copy()

print("Campaign table shape:", campaigns.shape)
print("Campaign columns:", campaigns.columns.tolist())

display(campaigns.head(10))

print("\nMissing values:")
display(campaigns.isna().sum().to_frame("missing_count"))

print("\nDuplicate campaign IDs:")
print(campaigns["campaign_id"].duplicated().sum())

Campaign table shape: (120, 7)
Campaign columns: ['campaign_id', 'campaign_name', 'channel', 'strategy_version', 'start_at', 'target_definition', 'end_at']


,campaign_id,campaign_name,channel,strategy_version,start_at,target_definition,end_at
0,CMP0000001,DIGITAL_FOLLOWUP,FIELD,legacy,2026-02-17 06:56:01,DPD>=30,2026-04-24 06:56:01
1,CMP0000002,BOUNCE,MIXED,v2,2026-04-30 17:51:31,DPD>=60,2026-05-18 17:51:31
2,CMP0000003,30DPD_W1,MIXED,v1,2026-04-11 16:02:51,DPD>=30,2026-05-18 16:02:51
3,CMP0000004,BOUNCE,VOICE,legacy,2026-04-28 06:15:30,DPD>=60,2026-05-29 06:15:30
4,CMP0000005,60DPD_INTENT,WHATSAPP,legacy,2026-05-04 04:16:21,DPD>=60,2026-07-08 04:16:21
5,CMP0000006,BOUNCE,MIXED,legacy,2026-01-23 09:30:10,DPD>=60,2026-04-13 09:30:10
6,CMP0000007,DIGITAL_FOLLOWUP,MIXED,v2,2026-02-14 15:20:05,PROMISE_BROKEN,2026-04-22 15:20:05
7,CMP0000008,NPA_RECOVERY,SMS,v2,2026-05-21 10:58:25,PROMISE_BROKEN,2026-06-27 10:58:25
8,CMP0000009,60DPD_INTENT,WHATSAPP,v1,2026-04-18 15:47:29,HIGH_RISK,2026-06-07 15:47:29
9,CMP0000010,30DPD_W1,FIELD,v1,2026-02-14 13:09:24,DPD>=30,2026-04-30 13:09:24



Missing values:


,missing_count
campaign_id,0
campaign_name,0
channel,0
strategy_version,0
start_at,0
target_definition,0
end_at,0



Duplicate campaign IDs:
0


In [33]:
campaigns = data["campaigns"].copy()

# Convert campaign dates to datetime
campaigns["start_at"] = pd.to_datetime(campaigns["start_at"], errors="coerce")
campaigns["end_at"] = pd.to_datetime(campaigns["end_at"], errors="coerce")

print("Campaigns by strategy version:")
display(
    campaigns["strategy_version"]
    .value_counts(dropna=False)
    .to_frame("campaign_count")
)

print("\nStrategy version by channel:")
display(
    pd.crosstab(
        campaigns["strategy_version"],
        campaigns["channel"],
        margins=True
    )
)

print("\nCampaigns by start month and strategy:")
campaigns["start_month"] = campaigns["start_at"].dt.to_period("M")

display(
    pd.crosstab(
        campaigns["start_month"],
        campaigns["strategy_version"],
        margins=True
    )
)

print("\nCampaign duration:")
campaigns["duration_days"] = (
    campaigns["end_at"] - campaigns["start_at"]
).dt.total_seconds() / 86400

display(campaigns["duration_days"].describe().to_frame("value"))

print("\nCampaign names and target definitions:")
display(
    campaigns.groupby(
        ["campaign_name", "target_definition", "strategy_version"]
    ).size().reset_index(name="campaign_count")
    .sort_values("campaign_count", ascending=False)
)

Campaigns by strategy version:


,campaign_count
strategy_version,
legacy,37
v3,32
v2,27
v1,24



Strategy version by channel:


channel,FIELD,MIXED,SMS,VOICE,WHATSAPP,All
strategy_version,,,,,,
legacy,6,6,9,5,11,37
v1,4,5,4,4,7,24
v2,5,6,7,4,5,27
v3,3,6,8,7,8,32
All,18,23,28,20,31,120



Campaigns by start month and strategy:


strategy_version,legacy,v1,v2,v3,All
start_month,,,,,
2026-01,10,8,1,5,24
2026-02,4,3,8,4,19
2026-03,6,2,9,10,27
2026-04,11,6,2,11,30
2026-05,6,5,7,2,20
All,37,24,27,32,120



Campaign duration:


,value
count,120.000000
mean,42.458333
std,24.725687
min,6.000000
25%,20.000000
50%,40.000000
75%,65.250000
max,89.000000



Campaign names and target definitions:


,campaign_name,target_definition,strategy_version,campaign_count
4,30DPD_W1,DPD>=60,v2,3
2,30DPD_W1,DPD>=30,v3,3
5,30DPD_W1,DPD>=60,v3,3
64,NPA_RECOVERY,NPA,v1,3
30,BOUNCE,DPD>=60,v2,3
...,...,...,...,...
63,NPA_RECOVERY,NPA,legacy,1
65,NPA_RECOVERY,NPA,v2,1
66,NPA_RECOVERY,PROMISE_BROKEN,legacy,1
67,NPA_RECOVERY,PROMISE_BROKEN,v1,1


In [34]:
targeting = data["daily_targeting"].copy()
campaigns = data["campaigns"].copy()
payments = data["payments"].copy()

print("Campaign IDs in daily_targeting:")
print("Unique:", targeting["campaign_id"].nunique())
print("Missing:", targeting["campaign_id"].isna().sum())

print("\nCampaign IDs in campaigns:")
print("Unique:", campaigns["campaign_id"].nunique())

# Check whether every targeted campaign exists in the campaign table
target_campaign_check = targeting.merge(
    campaigns[["campaign_id", "campaign_name", "strategy_version",
               "channel", "target_definition"]],
    on="campaign_id",
    how="left",
    indicator=True
)

print("\nTargeting-to-campaign match:")
display(target_campaign_check["_merge"].value_counts().to_frame("rows"))

print("\nSample unmatched targeting rows:")
display(
    target_campaign_check.loc[
        target_campaign_check["_merge"] == "left_only"
    ].head(10)
)

print("\nPayment columns:")
print(payments.columns.tolist())

Campaign IDs in daily_targeting:
Unique: 120
Missing: 0

Campaign IDs in campaigns:
Unique: 120

Targeting-to-campaign match:


,rows
_merge,
both,45000
left_only,0
right_only,0



Sample unmatched targeting rows:


,target_id,account_id,campaign_id,target_date,priority,recommended_channel,status,campaign_name,strategy_version,channel,target_definition,_merge



Payment columns:
['payment_id', 'account_id', 'borrower_id', 'event_at', 'payment_reference', 'amount', 'payment_status', 'payment_method', 'provider_id']


In [35]:
import pandas as pd

targeting = data["daily_targeting"].copy()
campaigns = data["campaigns"].copy()
payments = payments_consolidated.copy()

# Convert dates
targeting["target_date"] = pd.to_datetime(
    targeting["target_date"], errors="coerce"
)
payments["event_at"] = pd.to_datetime(
    payments["event_at"], errors="coerce"
)

# Add campaign strategy details to targeting records
targeting_campaigns = targeting.merge(
    campaigns[
        ["campaign_id", "campaign_name", "strategy_version",
         "channel", "target_definition", "start_at", "end_at"]
    ],
    on="campaign_id",
    how="left",
    validate="many_to_one"
)

# Keep successful payments for the initial analysis
successful_payments = payments[
    payments["payment_status"].astype(str).str.upper().eq("SUCCESS")
].copy()

print("Targeting records:", len(targeting_campaigns))
print("Successful payment records:", len(successful_payments))
print("Accounts targeted:", targeting_campaigns["account_id"].nunique())
print("Accounts with successful payments:",
      successful_payments["account_id"].nunique())

# Check how many targeted accounts have any successful payment
targeted_accounts = set(targeting_campaigns["account_id"].dropna())
paid_accounts = set(successful_payments["account_id"].dropna())

print("\nTargeted accounts with a successful payment at any time:",
      len(targeted_accounts & paid_accounts))
print("Percentage of targeted accounts with a successful payment:",
      round(
          100 * len(targeted_accounts & paid_accounts) /
          len(targeted_accounts), 2
      ), "%")

print("\nPayment date range:")
print(successful_payments["event_at"].min(),
      "to", successful_payments["event_at"].max())

print("\nTargeting date range:")
print(targeting_campaigns["target_date"].min(),
      "to", targeting_campaigns["target_date"].max())

Targeting records: 45000
Successful payment records: 17534
Accounts targeted: 23344
Accounts with successful payments: 13284

Targeted accounts with a successful payment at any time: 10288
Percentage of targeted accounts with a successful payment: 44.07 %

Payment date range:
2026-01-01 00:14:40 to 2026-08-08 23:50:23

Targeting date range:
2026-01-01 00:00:00 to 2026-08-08 00:00:00


In [36]:
import pandas as pd

targeting = data["daily_targeting"].copy()
campaigns = data["campaigns"].copy()
payments = payments_consolidated.copy()

# Prepare dates and filter successful payments
targeting["target_date"] = pd.to_datetime(
    targeting["target_date"], errors="coerce"
)
payments["event_at"] = pd.to_datetime(
    payments["event_at"], errors="coerce"
)

successful = payments[
    payments["payment_status"].astype(str).str.upper().eq("SUCCESS")
    & payments["event_at"].notna()
].copy()

# Attach campaign details
targeting = targeting.merge(
    campaigns[
        ["campaign_id", "campaign_name", "strategy_version",
         "channel", "target_definition"]
    ],
    on="campaign_id",
    how="left",
    validate="many_to_one"
)

# Match payments to targeting records on the same account
pairs = successful.merge(
    targeting[
        ["target_id", "account_id", "target_date", "campaign_id",
         "campaign_name", "strategy_version", "recommended_channel",
         "target_definition"]
    ],
    on="account_id",
    how="inner"
)

# Keep payments occurring from targeting date through the next 30 days
pairs["days_after_target"] = (
    pairs["event_at"] - pairs["target_date"]
).dt.total_seconds() / 86400

eligible = pairs[
    pairs["days_after_target"].between(0, 30, inclusive="both")
].copy()

# Assign each payment to its most recent eligible targeting record
attributed = (
    eligible.sort_values(["payment_id", "target_date", "target_id"])
    .drop_duplicates(subset="payment_id", keep="last")
)

print("Successful payments in source:", successful["payment_id"].nunique())
print("Payments with a targeting record in the preceding 30 days:",
      attributed["payment_id"].nunique())
print("Unique targeted accounts with an attributed payment:",
      attributed["account_id"].nunique())

print("\nAttributed payments by strategy version:")
strategy_summary = (
    attributed.groupby("strategy_version")
    .agg(
        payment_count=("payment_id", "nunique"),
        accounts_with_payments=("account_id", "nunique"),
        gross_successful_amount=("amount", "sum")
    )
    .sort_values("gross_successful_amount", ascending=False)
)

strategy_summary["gross_amount_crore"] = (
    strategy_summary["gross_successful_amount"] / 10_000_000
).round(2)

display(strategy_summary)

print("\nDays between targeting and payment:")
display(attributed["days_after_target"].describe().to_frame("days"))

Successful payments in source: 17534
Payments with a targeting record in the preceding 30 days: 3012
Unique targeted accounts with an attributed payment: 2806

Attributed payments by strategy version:


,payment_count,accounts_with_payments,gross_successful_amount,gross_amount_crore
strategy_version,,,,
legacy,922,880,69960063.27,7.00
v3,798,770,58766090.57,5.88
v2,679,650,52065709.29,5.21
v1,613,590,46771029.64,4.68



Days between targeting and payment:


,days
count,3012.000000
mean,13.940535
std,8.645114
min,0.017326
25%,6.386392
50%,13.494271
75%,21.312613
max,29.987708


In [37]:
payments = payments_consolidated.copy()

payments["event_at"] = pd.to_datetime(
    payments["event_at"], errors="coerce"
)

payments["month"] = payments["event_at"].dt.to_period("M")

successful = payments[
    payments["payment_status"].astype(str).str.upper().eq("SUCCESS")
].copy()

monthly = (
    successful.groupby("month")
    .agg(
        successful_transactions=("payment_id", "nunique"),
        gross_successful_amount=("amount", "sum"),
        accounts_with_payments=("account_id", "nunique")
    )
    .sort_index()
)

monthly["amount_crore"] = (
    monthly["gross_successful_amount"] / 10_000_000
)

monthly["amount_mom_change_pct"] = (
    monthly["gross_successful_amount"].pct_change() * 100
)

monthly["transaction_mom_change_pct"] = (
    monthly["successful_transactions"].pct_change() * 100
)

display(monthly.round(2))

,successful_transactions,gross_successful_amount,accounts_with_payments,amount_crore,amount_mom_change_pct,transaction_mom_change_pct
month,,,,,,
2026-01,2464,1.872291e+08,2374,18.72,NaN,NaN
2026-02,2268,1.701425e+08,2173,17.01,-9.13,-7.95
2026-03,2524,1.889124e+08,2419,18.89,11.03,11.29
2026-04,2406,1.751380e+08,2304,17.51,-7.29,-4.68
2026-05,2449,1.842503e+08,2344,18.43,5.20,1.79
2026-06,2366,1.755597e+08,2286,17.56,-4.72,-3.39
2026-07,2441,1.872423e+08,2335,18.72,6.65,3.17
2026-08,616,4.710970e+07,608,4.71,-74.84,-74.76


## 7. Payment status timelines and attribution

Compare payment status events and available interaction references. Do not assign a payment to the latest campaign or interaction without defensible attribution logic.

In [38]:
payments = payments_consolidated.copy()

payments["event_at"] = pd.to_datetime(
    payments["event_at"], errors="coerce"
)
payments["month"] = payments["event_at"].dt.to_period("M")
payments["payment_status"] = (
    payments["payment_status"].astype(str).str.upper()
)

# Summarise each payment status by month
monthly_status = (
    payments.pivot_table(
        index="month",
        columns="payment_status",
        values="amount",
        aggfunc="sum",
        fill_value=0
    )
    .sort_index()
)

# Calculate gross successful amounts and recorded reversal amounts
monthly_status["gross_success_crore"] = (
    monthly_status.get("SUCCESS", 0) / 10_000_000
)
monthly_status["reversed_crore"] = (
    monthly_status.get("REVERSED", 0) / 10_000_000
)

# Provisional measure only: subtract reversals recorded in the same month
monthly_status["provisional_net_crore"] = (
    monthly_status["gross_success_crore"]
    - monthly_status["reversed_crore"]
)

monthly_status["provisional_net_mom_pct"] = (
    monthly_status["provisional_net_crore"].pct_change() * 100
)

display(
    monthly_status[
        [
            "gross_success_crore",
            "reversed_crore",
            "provisional_net_crore",
            "provisional_net_mom_pct"
        ]
    ].round(2)
)

payment_status,gross_success_crore,reversed_crore,provisional_net_crore,provisional_net_mom_pct
month,,,,
2026-01,18.72,1.16,17.56,NaN
2026-02,17.01,1.15,15.87,-9.64
2026-03,18.89,1.44,17.45,10.01
2026-04,17.51,1.33,16.18,-7.30
2026-05,18.43,1.27,17.16,6.02
2026-06,17.56,1.29,16.26,-5.22
2026-07,18.72,1.53,17.20,5.76
2026-08,4.71,0.30,4.42,-74.33


In [39]:
payments = data["payments"].copy()

print("Payment table columns:")
print(payments.columns.tolist())

print("\nSample reversed transactions:")
display(
    payments[
        payments["payment_status"].astype(str).str.upper().eq("REVERSED")
    ].head(10)
)

print("\nSample successful transactions:")
display(
    payments[
        payments["payment_status"].astype(str).str.upper().eq("SUCCESS")
    ].head(10)
)

print("\nPayment reference completeness by status:")
display(
    payments.assign(
        status=payments["payment_status"].astype(str).str.upper(),
        reference_missing=payments["payment_reference"].isna()
    )
    .groupby("status")
    .agg(
        transaction_rows=("payment_id", "size"),
        missing_references=("reference_missing", "sum"),
        unique_accounts=("account_id", "nunique")
    )
)

Payment table columns:
['payment_id', 'account_id', 'borrower_id', 'event_at', 'payment_reference', 'amount', 'payment_status', 'payment_method', 'provider_id']

Sample reversed transactions:


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
3,PAYMENT0000004,ACC0017531,BRW0006618,2026-06-16 02:35:21,TXN0000031781,145082.17,REVERSED,CASH,VND0000013
11,PAYMENT0000012,ACC0008138,BRW0000540,2026-03-13 17:05:25,TXN0000046491,116971.89,REVERSED,NETBANKING,VND0000001
19,PAYMENT0000020,ACC0022251,BRW0011071,2026-01-10 19:06:03,TXN0000027006,144661.69,REVERSED,CARD,VND0000003
27,PAYMENT0000028,ACC0028651,BRW0011101,2026-05-24 23:43:17,TXN0000028567,74205.30,REVERSED,CARD,VND0000001
36,PAYMENT0000037,ACC0019433,BRW0000459,2026-03-29 07:09:14,TXN0000031615,26255.04,REVERSED,NACH,VND0000002
45,PAYMENT0000046,ACC0028983,BRW0010609,2026-04-17 06:26:52,TXN0000042379,125117.22,REVERSED,NACH,VND0000015
64,PAYMENT0000065,ACC0028084,BRW0008200,2026-07-07 15:59:57,TXN0000020314,24854.56,REVERSED,UPI,VND0000007
91,PAYMENT0000092,ACC0005651,BRW0010962,2026-03-19 19:10:23,TXN0000020456,132110.67,REVERSED,UPI,VND0000013
121,PAYMENT0000122,ACC0018554,BRW0007823,2026-06-14 05:41:54,TXN0000004794,87572.57,REVERSED,CASH,VND0000001
159,PAYMENT0000160,ACC0005979,BRW0008073,2026-05-01 22:11:32,TXN0000041970,103047.14,REVERSED,CASH,VND0000014



Sample successful transactions:


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
4,PAYMENT0000005,ACC0001770,BRW0011492,2026-03-03 06:08:23,TXN0000059257,115148.46,SUCCESS,NETBANKING,VND0000013
5,PAYMENT0000006,ACC0002408,BRW0009136,2026-05-25 10:59:51,TXN0000052202,142468.84,SUCCESS,CARD,VND0000012
6,PAYMENT0000007,ACC0017147,BRW0004710,2026-05-08 11:45:34,TXN0000050767,79641.72,SUCCESS,NETBANKING,VND0000008
7,PAYMENT0000008,ACC0027360,BRW0008644,2026-01-16 08:37:42,TXN0000061682,80638.15,SUCCESS,CARD,VND0000009
8,PAYMENT0000009,ACC0005092,BRW0003019,2026-03-31 08:11:38,TXN0000046720,75028.44,SUCCESS,CASH,VND0000003
9,PAYMENT0000010,ACC0011343,BRW0010947,2026-07-20 20:51:33,TXN0000031163,80004.84,SUCCESS,UPI,VND0000005
10,PAYMENT0000011,ACC0022813,BRW0011517,2026-06-13 10:49:29,TXN0000066098,27561.35,SUCCESS,UPI,VND0000010
12,PAYMENT0000013,ACC0026504,BRW0009822,2026-05-06 22:30:42,NaN,41543.11,SUCCESS,CARD,VND0000013
13,PAYMENT0000014,ACC0017077,BRW0011703,2026-03-08 16:59:47,TXN0000065948,12762.88,SUCCESS,NETBANKING,VND0000009
14,PAYMENT0000015,ACC0003730,BRW0000271,2026-01-25 08:23:30,TXN0000012250,15237.53,SUCCESS,CARD,VND0000003



Payment reference completeness by status:


,transaction_rows,missing_references,unique_accounts
status,,,
FAILED,3744,55,3443
PENDING,2592,39,2430
REVERSED,1284,23,1230
SUCCESS,17880,265,13284


In [40]:
payments = data["payments"].copy()

payments["payment_status"] = (
    payments["payment_status"].astype(str).str.upper()
)

successful_refs = set(
    payments.loc[
        payments["payment_status"] == "SUCCESS",
        "payment_reference"
    ].dropna()
)

reversed_refs = set(
    payments.loc[
        payments["payment_status"] == "REVERSED",
        "payment_reference"
    ].dropna()
)

common_refs = successful_refs.intersection(reversed_refs)

print("Unique SUCCESS references:", len(successful_refs))
print("Unique REVERSED references:", len(reversed_refs))
print("References shared by SUCCESS and REVERSED:", len(common_refs))

print("\nExamples of shared references:")
display(
    payments[
        payments["payment_reference"].isin(list(common_refs)[:10])
    ].sort_values("payment_reference")
)

Unique SUCCESS references: 15349
Unique REVERSED references: 1224
References shared by SUCCESS and REVERSED: 273

Examples of shared references:


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
12378,PAYMENT0012379,ACC0028546,BRW0004116,2026-02-10 08:05:52,TXN0000013942,92762.16,SUCCESS,CARD,VND0000014
5710,PAYMENT0005711,ACC0018131,BRW0010290,2026-04-30 06:02:37,TXN0000013942,71001.51,REVERSED,UPI,VND0000008
731,PAYMENT0000732,ACC0016155,BRW0002437,2026-02-02 18:05:35,TXN0000019705,93161.93,REVERSED,NETBANKING,VND0000011
24938,PAYMENT0024939,ACC0012972,BRW0007734,2026-01-30 15:47:36,TXN0000019705,39027.65,SUCCESS,NACH,VND0000009
4378,PAYMENT0004379,ACC0027720,BRW0005039,2026-03-07 13:19:52,TXN0000027954,51411.35,REVERSED,NACH,VND0000002
23319,PAYMENT0023320,ACC0015018,BRW0006686,2026-01-18 11:27:01,TXN0000027954,118124.35,SUCCESS,CARD,VND0000009
6391,PAYMENT0006392,ACC0004446,BRW0011216,2026-02-06 06:32:00,TXN0000027954,144255.55,SUCCESS,CARD,VND0000014
899,PAYMENT0000900,ACC0013817,BRW0002414,2026-06-06 10:57:25,TXN0000027954,2538.01,SUCCESS,CASH,VND0000007
19003,PAYMENT0019004,ACC0002164,BRW0011792,2026-07-10 07:42:07,TXN0000039862,125144.63,REVERSED,UPI,VND0000011
11903,PAYMENT0011904,ACC0025311,BRW0009667,2026-02-13 18:18:27,TXN0000039862,129229.89,SUCCESS,UPI,VND0000014


In [41]:
payments = data["payments"].copy()

payments["payment_status"] = (
    payments["payment_status"].astype(str).str.upper()
)

# Keep records with a reference shared by SUCCESS and REVERSED
success_refs = set(
    payments.loc[
        payments["payment_status"] == "SUCCESS",
        "payment_reference"
    ].dropna()
)

reversed_refs = set(
    payments.loc[
        payments["payment_status"] == "REVERSED",
        "payment_reference"
    ].dropna()
)

common_refs = success_refs & reversed_refs

matched = payments[
    payments["payment_reference"].isin(common_refs)
    & payments["payment_status"].isin(["SUCCESS", "REVERSED"])
].copy()

reference_check = (
    matched.groupby("payment_reference")
    .agg(
        statuses=("payment_status", "nunique"),
        accounts=("account_id", "nunique"),
        borrowers=("borrower_id", "nunique"),
        amounts=("amount", "nunique"),
        payment_records=("payment_id", "nunique")
    )
    .reset_index()
)

print("Shared references examined:", len(reference_check))

print("\nShared references linked to multiple accounts:")
print((reference_check["accounts"] > 1).sum())

print("Shared references linked to multiple borrowers:")
print((reference_check["borrowers"] > 1).sum())

print("Shared references with different amounts:")
print((reference_check["amounts"] > 1).sum())

print("\nExamples where account or amount differs:")
display(
    reference_check[
        (reference_check["accounts"] > 1)
        | (reference_check["amounts"] > 1)
    ].head(10)
)

Shared references examined: 273

Shared references linked to multiple accounts:
273
Shared references linked to multiple borrowers:
273
Shared references with different amounts:
273

Examples where account or amount differs:


,payment_reference,statuses,accounts,borrowers,amounts,payment_records
0,TXN0000000113,2,3,3,3,3
1,TXN0000000568,2,2,2,2,2
2,TXN0000000635,2,2,2,2,2
3,TXN0000001233,2,2,2,2,2
4,TXN0000001555,2,2,2,2,2
5,TXN0000001743,2,2,2,2,2
6,TXN0000001845,2,2,2,2,2
7,TXN0000001903,2,2,2,2,2
8,TXN0000002005,2,2,2,2,2
9,TXN0000002126,2,2,2,2,2


In [42]:
payments = data["payments"].copy()

payments["event_at"] = pd.to_datetime(payments["event_at"])
payments["month"] = payments["event_at"].dt.to_period("M").astype(str)
payments["payment_status"] = (
    payments["payment_status"].astype(str).str.upper()
)

monthly_payments = (
    payments.groupby(["month", "payment_status"])
    .agg(
        transactions=("payment_id", "nunique"),
        amount=("amount", "sum")
    )
    .reset_index()
)

monthly_summary = monthly_payments.pivot(
    index="month",
    columns="payment_status",
    values="amount"
).fillna(0)

monthly_summary["gross_success_cr"] = (
    monthly_summary.get("SUCCESS", 0) / 10_000_000
)

monthly_summary["reversed_cr"] = (
    monthly_summary.get("REVERSED", 0) / 10_000_000
)

display(
    monthly_summary[
        ["gross_success_cr", "reversed_cr"]
    ].round(2)
)

payment_status,gross_success_cr,reversed_cr
month,,
2026-01,19.11,1.18
2026-02,17.41,1.18
2026-03,19.32,1.49
2026-04,17.84,1.38
2026-05,18.70,1.34
2026-06,17.87,1.34
2026-07,19.03,1.55
2026-08,4.85,0.30


In [43]:
monthly_summary = monthly_summary.sort_index().copy()

monthly_summary["success_mom_pct"] = (
    monthly_summary["gross_success_cr"].pct_change() * 100
)

monthly_summary["reversed_mom_pct"] = (
    monthly_summary["reversed_cr"].pct_change() * 100
)

display(
    monthly_summary[
        ["gross_success_cr", "success_mom_pct",
         "reversed_cr", "reversed_mom_pct"]
    ].round(2)
)

payment_status,gross_success_cr,success_mom_pct,reversed_cr,reversed_mom_pct
month,,,,
2026-01,19.11,NaN,1.18,NaN
2026-02,17.41,-8.91,1.18,-0.35
2026-03,19.32,10.99,1.49,26.58
2026-04,17.84,-7.66,1.38,-7.60
2026-05,18.70,4.83,1.34,-2.83
2026-06,17.87,-4.45,1.34,-0.00
2026-07,19.03,6.46,1.55,15.75
2026-08,4.85,-74.49,0.30,-80.41


In [44]:
print("Original payments rows:", len(data["payments"]))

print(
    "Original unique payment IDs:",
    data["payments"]["payment_id"].nunique()
)

print("Consolidated payments rows:", len(payments_consolidated))

print(
    "Consolidated unique payment IDs:",
    payments_consolidated["payment_id"].nunique()
)

Original payments rows: 25500
Original unique payment IDs: 25000
Consolidated payments rows: 25000
Consolidated unique payment IDs: 25000


In [45]:
payments_clean = payments_consolidated.copy()

payments_clean["event_at"] = pd.to_datetime(
    payments_clean["event_at"]
)

payments_clean["payment_status"] = (
    payments_clean["payment_status"].astype(str).str.upper()
)

payments_clean["month"] = (
    payments_clean["event_at"].dt.to_period("M").astype(str)
)

monthly_clean = (
    payments_clean
    .groupby(["month", "payment_status"])["amount"]
    .sum()
    .unstack(fill_value=0)
    .sort_index()
)

monthly_clean["gross_success_cr"] = (
    monthly_clean["SUCCESS"] / 10_000_000
)

monthly_clean["reversed_cr"] = (
    monthly_clean["REVERSED"] / 10_000_000
)

monthly_clean["success_mom_pct"] = (
    monthly_clean["gross_success_cr"].pct_change() * 100
)

display(
    monthly_clean[
        ["gross_success_cr", "reversed_cr", "success_mom_pct"]
    ].round(2)
)

payment_status,gross_success_cr,reversed_cr,success_mom_pct
month,,,
2026-01,18.72,1.16,NaN
2026-02,17.01,1.15,-9.13
2026-03,18.89,1.44,11.03
2026-04,17.51,1.33,-7.29
2026-05,18.43,1.27,5.20
2026-06,17.56,1.29,-4.72
2026-07,18.72,1.53,6.65
2026-08,4.71,0.30,-74.84


In [46]:
accounts = data["accounts"].copy()

accounts["dpd_bucket"] = pd.cut(
    accounts["dpd"],
    bins=[-1, 0, 30, 60, 90, 180],
    labels=[
        "0 days",
        "1–30 days",
        "31–60 days",
        "61–90 days",
        "91–180 days"
    ]
)

dpd_analysis = (
    accounts.groupby("dpd_bucket", observed=False)
    .agg(
        accounts=("account_id", "nunique"),
        outstanding_amount=("outstanding_amount", "sum"),
        average_dpd=("dpd", "mean")
    )
    .reset_index()
)

dpd_analysis["outstanding_cr"] = (
    dpd_analysis["outstanding_amount"] / 10_000_000
)

display(
    dpd_analysis[
        ["dpd_bucket", "accounts", "outstanding_cr", "average_dpd"]
    ].round(2)
)

,dpd_bucket,accounts,outstanding_cr,average_dpd
0,0 days,2685,94.82,0.00
1,1–30 days,10880,383.41,12.73
2,31–60 days,5514,191.45,52.54
3,61–90 days,5468,189.59,82.48
4,91–180 days,5453,189.64,149.64


In [47]:
accounts = data["accounts"].copy()

payments_clean = payments_consolidated.copy()
payments_clean["payment_status"] = (
    payments_clean["payment_status"].astype(str).str.upper()
)

# Sum successful payments by account
successful_by_account = (
    payments_clean[
        payments_clean["payment_status"] == "SUCCESS"
    ]
    .groupby("account_id")
    .agg(
        successful_payment_amount=("amount", "sum"),
        successful_payment_count=("payment_id", "nunique")
    )
    .reset_index()
)

# Join payment outcomes to the account snapshot
account_payment_analysis = accounts.merge(
    successful_by_account,
    on="account_id",
    how="left"
)

account_payment_analysis[
    ["successful_payment_amount", "successful_payment_count"]
] = account_payment_analysis[
    ["successful_payment_amount", "successful_payment_count"]
].fillna(0)

account_payment_analysis["dpd_bucket"] = pd.cut(
    account_payment_analysis["dpd"],
    bins=[-1, 0, 30, 60, 90, 180],
    labels=["0 days", "1–30 days", "31–60 days", "61–90 days", "91–180 days"]
)

bucket_payment_analysis = (
    account_payment_analysis
    .groupby("dpd_bucket", observed=False)
    .agg(
        accounts=("account_id", "nunique"),
        accounts_with_successful_payment=(
            "successful_payment_count",
            lambda x: (x > 0).sum()
        ),
        successful_payment_amount=(
            "successful_payment_amount", "sum"
        ),
        outstanding_amount=("outstanding_amount", "sum")
    )
    .reset_index()
)

bucket_payment_analysis["payment_account_rate_pct"] = (
    bucket_payment_analysis["accounts_with_successful_payment"]
    / bucket_payment_analysis["accounts"] * 100
)

bucket_payment_analysis["successful_payment_cr"] = (
    bucket_payment_analysis["successful_payment_amount"] / 10_000_000
)

display(
    bucket_payment_analysis[
        [
            "dpd_bucket",
            "accounts",
            "accounts_with_successful_payment",
            "payment_account_rate_pct",
            "successful_payment_cr"
        ]
    ].round(2)
)

,dpd_bucket,accounts,accounts_with_successful_payment,payment_account_rate_pct,successful_payment_cr
0,0 days,2685,1220,45.44,11.75
1,1–30 days,10880,4719,43.37,47.15
2,31–60 days,5514,2524,45.77,25.28
3,61–90 days,5468,2436,44.55,23.87
4,91–180 days,5453,2385,43.74,23.51


In [48]:
targeting = data["daily_targeting"].copy()

print("Targeting records:", len(targeting))

print("\nTargeting status by recommended channel:")
display(
    pd.crosstab(
        targeting["recommended_channel"],
        targeting["status"],
        margins=True
    )
)

print("\nTargeting volume by channel:")
display(
    targeting["recommended_channel"]
    .value_counts()
    .rename_axis("channel")
    .reset_index(name="targeting_records")
)

Targeting records: 45000

Targeting status by recommended channel:


status,CONTACTED,EXPIRED,QUEUED,SKIPPED,All
recommended_channel,,,,,
FIELD,2842,2933,2787,2803,11365
SMS,2822,2744,2848,2807,11221
VOICE,2780,2911,2786,2725,11202
WHATSAPP,2810,2783,2781,2838,11212
All,11254,11371,11202,11173,45000



Targeting volume by channel:


,channel,targeting_records
0,FIELD,11365
1,SMS,11221
2,WHATSAPP,11212
3,VOICE,11202


In [49]:
targeting = data["daily_targeting"].copy()
campaigns = data["campaigns"].copy()

targeting["target_date"] = pd.to_datetime(
    targeting["target_date"]
)

payments_clean = payments_consolidated.copy()
payments_clean["event_at"] = pd.to_datetime(
    payments_clean["event_at"]
)
payments_clean["payment_status"] = (
    payments_clean["payment_status"].astype(str).str.upper()
)

successful = payments_clean[
    payments_clean["payment_status"] == "SUCCESS"
].copy()

# Add campaign channel to each targeting record
targeting = targeting.merge(
    campaigns[["campaign_id", "channel"]],
    on="campaign_id",
    how="left",
    suffixes=("", "_campaign")
)

# Use the recommended channel recorded in daily_targeting
targeting = targeting.rename(
    columns={"recommended_channel": "target_channel"}
)

# Match successful payments to the most recent targeting record
# for the same account within the preceding 30 days.
matches = successful.merge(
    targeting[
        ["account_id", "target_id", "target_date", "target_channel"]
    ],
    on="account_id",
    how="inner"
)

matches["days_after_target"] = (
    matches["event_at"] - matches["target_date"]
).dt.total_seconds() / 86400

matches = matches[
    matches["days_after_target"].between(0, 30, inclusive="both")
].copy()

matches = (
    matches.sort_values("target_date")
    .drop_duplicates("payment_id", keep="last")
)

channel_results = (
    matches.groupby("target_channel")
    .agg(
        attributed_payments=("payment_id", "nunique"),
        accounts_with_attributed_payments=("account_id", "nunique"),
        attributed_amount=("amount", "sum")
    )
    .reset_index()
)

channel_results["attributed_amount_cr"] = (
    channel_results["attributed_amount"] / 10_000_000
)

display(
    channel_results[
        [
            "target_channel",
            "attributed_payments",
            "accounts_with_attributed_payments",
            "attributed_amount_cr"
        ]
    ].sort_values("attributed_amount_cr", ascending=False).round(2)
)

,target_channel,attributed_payments,accounts_with_attributed_payments,attributed_amount_cr
2,VOICE,784,752,6.01
3,WHATSAPP,735,701,5.61
1,SMS,759,723,5.58
0,FIELD,734,711,5.55


In [50]:
vendor_telephony = data["vendor_telephony"].copy()

print("Shape:", vendor_telephony.shape)

print("\nColumns:")
print(vendor_telephony.columns.tolist())

print("\nVendor telephony sample:")
display(vendor_telephony.head(10))

print("\nSummary:")
display(vendor_telephony.describe(include="all").T)

Shape: (15, 6)

Columns:
['vendor_id', 'vendor_name', 'vendor_account_id', 'timezone', 'status', 'schema_version']

Vendor telephony sample:


,vendor_id,vendor_name,vendor_account_id,timezone,status,schema_version
0,VND0000001,Airtel,VAC342762,Asia/Kolkata,INACTIVE,v3
1,VND0000002,Exotel,VAC456766,UTC,INACTIVE,v3
2,VND0000003,Twilio,VAC074211,UTC,INACTIVE,v3
3,VND0000004,Twilio,VAC321507,UTC,ACTIVE,v1
4,VND0000005,Twilio,VAC976733,Asia/Kolkata,ACTIVE,v1
5,VND0000006,Knowlarity,VAC544470,UTC,ACTIVE,v2
6,VND0000007,TataTele,VAC997592,UTC,INACTIVE,v1
7,VND0000008,Airtel,VAC496672,Asia/Kolkata,INACTIVE,v3
8,VND0000009,Exotel,VAC382568,Asia/Kolkata,INACTIVE,v1
9,VND0000010,Airtel,VAC172916,UTC,INACTIVE,v1



Summary:


,count,unique,top,freq
vendor_id,15,15,VND0000001,1
vendor_name,15,5,Airtel,5
vendor_account_id,15,15,VAC342762,1
timezone,15,2,UTC,8
status,15,2,INACTIVE,9
schema_version,15,3,v1,7


In [51]:
calls = data["calls"].copy()

print("Columns:")
print(calls.columns.tolist())

display(calls.head(5))

Columns:
['call_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'campaign_id', 'direction', 'vendor_id', 'call_status', 'duration_sec', 'timezone']


,call_id,account_id,borrower_id,event_at,agent_id,campaign_id,direction,vendor_id,call_status,duration_sec,timezone
0,CALL0000001,ACC0011505,BRW0007139,2026-07-15 15:36:22,AGT0000955,CMP0000015,OUTBOUND,VND0000013,NO_ANSWER,601,Asia/Dubai
1,CALL0000002,ACC0002025,BRW0006253,2026-06-10 06:48:27,AGT0000853,CMP0000060,OUTBOUND,VND0000001,ANSWERED,224,Asia/Kolkata
2,CALL0000003,ACC0013375,BRW0007663,2026-04-07 00:35:35,AGT0000525,CMP0000060,OUTBOUND,VND0000008,FAILED,7,Asia/Dubai
3,CALL0000004,ACC0021534,BRW0008873,2026-02-12 14:16:57,AGT0000945,CMP0000060,OUTBOUND,VND0000001,VOICEMAIL,896,Asia/Dubai
4,CALL0000005,ACC0018502,BRW0004996,2026-05-24 15:33:12,AGT0000966,CMP0000053,OUTBOUND,VND0000006,ANSWERED,711,Asia/Kolkata


## 8. Channel and agent operational analysis

Summarise call/channel outcomes and session information where the supplied fields support it. Rates depend on valid denominators and reliable identities.

In [52]:
calls = data["calls"].copy()
campaigns = data["campaigns"].copy()

# Attach campaign channel to each call
calls = calls.merge(
    campaigns[["campaign_id", "channel"]],
    on="campaign_id",
    how="left"
)

# Summarise call outcomes by channel
channel_calls = (
    calls.groupby("channel")
    .agg(
        total_calls=("call_id", "nunique"),
        answered_calls=("call_status", lambda x: (x == "ANSWERED").sum()),
        total_duration_sec=("duration_sec", "sum"),
        avg_duration_sec=("duration_sec", "mean")
    )
    .reset_index()
)

channel_calls["answer_rate_pct"] = (
    channel_calls["answered_calls"] /
    channel_calls["total_calls"] * 100
)

channel_calls["total_duration_hours"] = (
    channel_calls["total_duration_sec"] / 3600
)

display(
    channel_calls.sort_values(
        "answer_rate_pct", ascending=False
    ).round(2)
)

,channel,total_calls,answered_calls,total_duration_sec,avg_duration_sec,answer_rate_pct,total_duration_hours
1,MIXED,17082,3490,7848388,452.98,20.43,2180.11
4,WHATSAPP,23404,4763,10705412,451.04,20.35,2973.73
0,FIELD,13423,2704,6132961,449.66,20.14,1703.60
2,SMS,21062,4240,9628926,450.62,20.13,2674.70
3,VOICE,15029,2949,6897679,451.36,19.62,1916.02


In [53]:
display(
    channel_calls[
        [
            "channel",
            "total_calls",
            "answered_calls",
            "answer_rate_pct",
            "total_duration_hours",
            "avg_duration_sec"
        ]
    ].sort_values("answer_rate_pct", ascending=False).round(2)
)

,channel,total_calls,answered_calls,answer_rate_pct,total_duration_hours,avg_duration_sec
1,MIXED,17082,3490,20.43,2180.11,452.98
4,WHATSAPP,23404,4763,20.35,2973.73,451.04
0,FIELD,13423,2704,20.14,1703.60,449.66
2,SMS,21062,4240,20.13,2674.70,450.62
3,VOICE,15029,2949,19.62,1916.02,451.36


In [54]:
targeting = data["daily_targeting"].copy()
accounts = data["accounts"].copy()

targeting_accounts = targeting.merge(
    accounts[
        ["account_id", "risk_segment", "dpd", "outstanding_amount"]
    ],
    on="account_id",
    how="left"
)

channel_profile = (
    targeting_accounts.groupby("recommended_channel")
    .agg(
        targeting_records=("target_id", "nunique"),
        unique_accounts=("account_id", "nunique"),
        avg_dpd=("dpd", "mean"),
        avg_outstanding=("outstanding_amount", "mean"),
        high_risk_records=("risk_segment", lambda x: (x == "HIGH").sum())
    )
    .reset_index()
)

display(channel_profile.round(2))

,recommended_channel,targeting_records,unique_accounts,avg_dpd,avg_outstanding,high_risk_records
0,FIELD,11365,9493,56.54,351220.63,2876
1,SMS,11221,9360,56.43,350865.22,2746
2,VOICE,11202,9393,55.91,351208.54,2760
3,WHATSAPP,11212,9363,57.18,347460.81,2843


In [55]:
import pandas as pd

targeting = data["daily_targeting"].copy()
targeting["target_date"] = pd.to_datetime(targeting["target_date"])

accounts = data["accounts"].copy()

payments = payments_consolidated.copy()
payments["event_at"] = pd.to_datetime(payments["event_at"])

successful = payments[
    payments["payment_status"].str.upper().eq("SUCCESS")
].copy()

# Attach account risk information
targeting = targeting.merge(
    accounts[["account_id", "risk_segment"]],
    on="account_id",
    how="left"
)

# Match payments to targeting records for the same account
matches = successful.merge(
    targeting[
        ["account_id", "target_id", "target_date",
         "recommended_channel", "risk_segment"]
    ],
    on="account_id",
    how="inner"
)

matches["days_after_target"] = (
    matches["event_at"] - matches["target_date"]
).dt.total_seconds() / 86400

matches = matches[
    matches["days_after_target"].between(0, 30)
].copy()

# Attribute each payment to its most recent eligible targeting record
matches = (
    matches.sort_values("target_date")
    .drop_duplicates("payment_id", keep="last")
)

# Count unique targeted accounts and accounts with attributed payments
target_counts = (
    targeting.groupby(["recommended_channel", "risk_segment"])
    .agg(targeted_accounts=("account_id", "nunique"))
    .reset_index()
)

payment_counts = (
    matches.groupby(["recommended_channel", "risk_segment"])
    .agg(accounts_with_payments=("account_id", "nunique"))
    .reset_index()
)

comparison = target_counts.merge(
    payment_counts,
    on=["recommended_channel", "risk_segment"],
    how="left"
)

comparison["accounts_with_payments"] = (
    comparison["accounts_with_payments"].fillna(0)
)

comparison["payment_rate_pct"] = (
    comparison["accounts_with_payments"]
    / comparison["targeted_accounts"] * 100
)

display(
    comparison.sort_values(
        ["risk_segment", "payment_rate_pct"],
        ascending=[True, False]
    ).round(2)
)

,recommended_channel,risk_segment,targeted_accounts,accounts_with_payments,payment_rate_pct
8,VOICE,HIGH,2319,186,8.02
0,FIELD,HIGH,2395,181,7.56
4,SMS,HIGH,2302,162,7.04
12,WHATSAPP,HIGH,2361,160,6.78
5,SMS,LOW,2453,209,8.52
13,WHATSAPP,LOW,2343,187,7.98
9,VOICE,LOW,2370,185,7.81
1,FIELD,LOW,2357,165,7.00
14,WHATSAPP,MEDIUM,2355,194,8.24
10,VOICE,MEDIUM,2430,194,7.98


In [56]:
# Rebuild payment attribution with targeting status included

targeting_status = data["daily_targeting"].copy()
targeting_status["target_date"] = pd.to_datetime(
    targeting_status["target_date"]
)

payments_clean = payments_consolidated.copy()
payments_clean["event_at"] = pd.to_datetime(
    payments_clean["event_at"]
)

successful = payments_clean[
    payments_clean["payment_status"].str.upper().eq("SUCCESS")
].copy()

# Match successful payments to targeting records for the same account
status_matches = successful.merge(
    targeting_status[
        [
            "account_id",
            "target_id",
            "target_date",
            "recommended_channel",
            "status"
        ]
    ],
    on="account_id",
    how="inner"
)

status_matches["days_after_target"] = (
    status_matches["event_at"] - status_matches["target_date"]
).dt.total_seconds() / 86400

# Keep payments occurring within 30 days after targeting
status_matches = status_matches[
    status_matches["days_after_target"].between(0, 30)
].copy()

# Attribute each payment to its most recent eligible targeting record
status_matches = (
    status_matches.sort_values("target_date")
    .drop_duplicates("payment_id", keep="last")
)

print("Attributed successful payments:", status_matches["payment_id"].nunique())

display(
    status_matches.groupby(
        ["recommended_channel", "status"]
    ).agg(
        attributed_payments=("payment_id", "nunique"),
        accounts_with_payments=("account_id", "nunique"),
        attributed_amount=("amount", "sum")
    ).reset_index().round(2)
)

Attributed successful payments: 3012


,recommended_channel,status,attributed_payments,accounts_with_payments,attributed_amount
0,FIELD,CONTACTED,185,182,13795417.08
1,FIELD,EXPIRED,201,191,16133296.55
2,FIELD,QUEUED,173,172,12371699.71
3,FIELD,SKIPPED,175,170,13174536.23
4,SMS,CONTACTED,198,190,13794422.35
5,SMS,EXPIRED,180,176,12828487.07
6,SMS,QUEUED,202,195,15485478.51
7,SMS,SKIPPED,179,169,13726587.05
8,VOICE,CONTACTED,180,176,13417303.33
9,VOICE,EXPIRED,204,194,14859964.06


In [57]:
agents = data["agents"].copy()
agent_sessions = data["agent_sessions"].copy()

print("AGENTS")
print(agents.columns.tolist())
display(agents.head(5))

print("\nAGENT SESSIONS")
print(agent_sessions.columns.tolist())
display(agent_sessions.head(5))

AGENTS
['agent_id', 'employee_code', 'agent_name', 'vendor_id', 'team', 'status', 'joined_at', 'updated_at']


,agent_id,employee_code,agent_name,vendor_id,team,status,joined_at,updated_at
0,AGT0000760,EMP00323,Vikram Shah,VND0000012,T3,INACTIVE,2025-06-19 20:30:53,2025-12-11 03:40:22
1,AGT0000171,EMP00079,Pooja Nair,VND0000012,T1,INACTIVE,2025-08-26 11:06:26,2025-10-11 08:30:33
2,AGT0000766,EMP00030,Sneha Das,VND0000002,FIELD,SUSPENDED,2024-05-30 11:23:06,2026-05-16 03:02:02
3,AGT0000031,EMP00334,Aarav Sharma,VND0000007,T1,INACTIVE,2024-10-11 18:58:28,2025-10-16 13:31:21
4,AGT0000669,EMP00014,Aarav Sharma,VND0000001,T3,SUSPENDED,2024-09-01 19:55:06,2026-07-04 03:33:11



AGENT SESSIONS
['session_id', 'agent_id', 'login_at', 'channel', 'device_id', 'timezone', 'logout_at']


,session_id,agent_id,login_at,channel,device_id,timezone,logout_at
0,SES0000001,AGT0000117,2026-08-06 13:33:27,VOICE,DEV00243,Asia/Kolkata,2026-08-06 18:41:13
1,SES0000002,AGT0000847,2026-05-27 02:02:28,FIELD,DEV01359,UTC,2026-05-27 09:02:31
2,SES0000003,AGT0000989,2026-08-01 20:33:44,FIELD,DEV01432,Asia/Kolkata,2026-08-02 05:53:43
3,SES0000004,AGT0000961,2026-01-13 19:50:52,FIELD,DEV00402,Asia/Kolkata,2026-01-13 22:20:49
4,SES0000005,AGT0000280,2026-03-03 15:05:04,VOICE,DEV01288,UTC,2026-03-03 18:19:09


In [58]:
sessions = data["agent_sessions"].copy()

sessions["login_at"] = pd.to_datetime(sessions["login_at"])
sessions["logout_at"] = pd.to_datetime(sessions["logout_at"])

sessions["session_hours"] = (
    sessions["logout_at"] - sessions["login_at"]
).dt.total_seconds() / 3600

# Flag invalid or unusually long sessions for review
sessions["valid_session"] = (
    sessions["session_hours"].between(0, 24)
)

session_summary = (
    sessions.groupby("channel")
    .agg(
        total_sessions=("session_id", "nunique"),
        valid_sessions=("valid_session", "sum"),
        total_session_hours=(
            "session_hours",
            lambda x: x[sessions.loc[x.index, "valid_session"]].sum()
        ),
        avg_session_hours=(
            "session_hours",
            lambda x: x[sessions.loc[x.index, "valid_session"]].mean()
        )
    )
    .reset_index()
)

display(session_summary.round(2))

,channel,total_sessions,valid_sessions,total_session_hours,avg_session_hours
0,FIELD,3796,3796,19888.25,5.24
1,SMS,3654,3654,19258.04,5.27
2,VOICE,3740,3740,19769.37,5.29
3,WHATSAPP,3810,3810,19955.23,5.24


In [59]:
# Aggregate recorded agent session hours by channel
hours_by_channel = (
    sessions[
        sessions["valid_session"]
    ]
    .groupby("channel")["session_hours"]
    .sum()
    .reset_index(name="session_hours")
)

# Aggregate attributed payments by recommended channel
amount_by_channel = (
    matches.groupby("recommended_channel")
    .agg(
        attributed_amount=("amount", "sum"),
        attributed_payments=("payment_id", "nunique")
    )
    .reset_index()
)

# Join the two summaries
efficiency = amount_by_channel.merge(
    hours_by_channel,
    left_on="recommended_channel",
    right_on="channel",
    how="left"
)

efficiency["amount_per_session_hour"] = (
    efficiency["attributed_amount"] / efficiency["session_hours"]
)

efficiency["attributed_amount_cr"] = (
    efficiency["attributed_amount"] / 10_000_000
)

display(
    efficiency[
        [
            "recommended_channel",
            "attributed_payments",
            "attributed_amount_cr",
            "session_hours",
            "amount_per_session_hour"
        ]
    ].round(2)
)

,recommended_channel,attributed_payments,attributed_amount_cr,session_hours,amount_per_session_hour
0,FIELD,734,5.55,19888.25,2789.33
1,SMS,759,5.58,19258.04,2899.31
2,VOICE,784,6.01,19769.37,3041.95
3,WHATSAPP,735,5.61,19955.23,2812.07


In [60]:
print("Columns in payments_consolidated:")
print(payments_consolidated.columns.tolist())

display(payments_consolidated.head())

Columns in payments_consolidated:
['payment_id', 'account_id', 'borrower_id', 'event_at', 'payment_reference', 'amount', 'payment_status', 'payment_method', 'provider_id']


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
0,PAYMENT0000001,ACC0015539,BRW0011363,2026-02-27 01:28:12,TXN0000007457,22433.23,FAILED,CARD,VND0000001
1,PAYMENT0000002,ACC0004445,BRW0011306,2026-07-23 20:25:20,TXN0000030016,23295.11,FAILED,UPI,VND0000004
2,PAYMENT0000003,ACC0003853,BRW0002549,2026-01-11 21:27:46,TXN0000010466,118814.97,FAILED,NACH,VND0000013
3,PAYMENT0000004,ACC0017531,BRW0006618,2026-06-16 02:35:21,TXN0000031781,145082.17,REVERSED,CASH,VND0000013
4,PAYMENT0000005,ACC0001770,BRW0011492,2026-03-03 06:08:23,TXN0000059257,115148.46,SUCCESS,NETBANKING,VND0000013


In [61]:

payments_analysis = payments_consolidated.copy()

payments_analysis["event_at"] = pd.to_datetime(
    payments_analysis["event_at"]
)

payments_analysis["month"] = (
    payments_analysis["event_at"].dt.to_period("M")
)

# Keep successful payments only
successful_payments = payments_analysis[
    payments_analysis["payment_status"].eq("SUCCESS")
].copy()

# Summarise monthly payments
monthly_recovery = (
    successful_payments.groupby("month")
    .agg(
        successful_transactions=("payment_id", "nunique"),
        gross_successful_amount=("amount", "sum")
    )
    .reset_index()
)

# Convert rupees to crore
monthly_recovery["amount_crore"] = (
    monthly_recovery["gross_successful_amount"] / 10_000_000
)

# Calculate month-on-month change
monthly_recovery["mom_change_pct"] = (
    monthly_recovery["gross_successful_amount"]
    .pct_change() * 100
)

display(monthly_recovery.round(2))


,month,successful_transactions,gross_successful_amount,amount_crore,mom_change_pct
0,2026-01,2464,1.872291e+08,18.72,NaN
1,2026-02,2268,1.701425e+08,17.01,-9.13
2,2026-03,2524,1.889124e+08,18.89,11.03
3,2026-04,2406,1.751380e+08,17.51,-7.29
4,2026-05,2449,1.842503e+08,18.43,5.20
5,2026-06,2366,1.755597e+08,17.56,-4.72
6,2026-07,2441,1.872423e+08,18.72,6.65
7,2026-08,616,4.710970e+07,4.71,-74.84


In [62]:
# Examine payment statuses
print("Payment status counts:")
display(
    payments_consolidated["payment_status"]
    .value_counts()
    .rename_axis("payment_status")
    .reset_index(name="transactions")
)

# Check whether payment references are reused
reference_summary = (
    payments_consolidated.dropna(subset=["payment_reference"])
    .groupby("payment_reference")
    .agg(
        payment_ids=("payment_id", "nunique"),
        accounts=("account_id", "nunique"),
        statuses=("payment_status", "nunique")
    )
    .reset_index()
)

reused_references = reference_summary[
    reference_summary["payment_ids"] > 1
]

print("Reused references:", len(reused_references))
print(
    "References associated with multiple statuses:",
    (reference_summary["statuses"] > 1).sum()
)

display(reused_references.head(10))

Payment status counts:


,payment_status,transactions
0,SUCCESS,17534
1,FAILED,3677
2,PENDING,2535
3,REVERSED,1254


Reused references: 3407
References associated with multiple statuses: 1681


,payment_reference,payment_ids,accounts,statuses
5,TXN0000000027,2,2,2
8,TXN0000000032,3,3,2
28,TXN0000000113,3,3,2
35,TXN0000000132,2,2,1
36,TXN0000000134,2,2,1
39,TXN0000000154,2,2,1
50,TXN0000000186,2,2,1
53,TXN0000000196,2,2,1
58,TXN0000000212,2,2,1
71,TXN0000000271,2,2,1


In [63]:
# Separate successful and reversed payments
successful = payments_consolidated[
    payments_consolidated["payment_status"] == "SUCCESS"
].copy()

reversed_payments = payments_consolidated[
    payments_consolidated["payment_status"] == "REVERSED"
].copy()

# Compare reversal records with successful payments
# using account ID and amount as candidate matching fields
candidate_matches = reversed_payments.merge(
    successful,
    on=["account_id", "amount"],
    how="left",
    suffixes=("_reversed", "_success")
)

# Summarise candidate matches
match_summary = (
    candidate_matches.groupby("payment_id_reversed")
    .agg(
        candidate_success_matches=("payment_id_success", "count")
    )
    .reset_index()
)

print("Reversed payment records:", len(reversed_payments))
print(
    "Reversals with at least one candidate match:",
    (match_summary["candidate_success_matches"] > 0).sum()
)
print(
    "Reversals with multiple candidate matches:",
    (match_summary["candidate_success_matches"] > 1).sum()
)

display(match_summary.head(10))

Reversed payment records: 1254
Reversals with at least one candidate match: 0
Reversals with multiple candidate matches: 0


,payment_id_reversed,candidate_success_matches
0,PAYMENT0000004,0
1,PAYMENT0000012,0
2,PAYMENT0000020,0
3,PAYMENT0000028,0
4,PAYMENT0000037,0
5,PAYMENT0000046,0
6,PAYMENT0000065,0
7,PAYMENT0000092,0
8,PAYMENT0000122,0
9,PAYMENT0000160,0


In [64]:
# Find account-and-amount combinations shared by SUCCESS and REVERSED
success_keys = successful[
    ["account_id", "amount"]
].drop_duplicates()

reversed_keys = reversed_payments[
    ["account_id", "amount"]
].drop_duplicates()

overlap = reversed_keys.merge(
    success_keys,
    on=["account_id", "amount"],
    how="inner"
)

print("Unique reversed account-amount combinations:", len(reversed_keys))
print("Unique successful account-amount combinations:", len(success_keys))
print("Exact account-and-amount overlaps:", len(overlap))

display(overlap.head(10))

Unique reversed account-amount combinations: 1254
Unique successful account-amount combinations: 17534
Exact account-and-amount overlaps: 0


,account_id,amount


In [65]:
# Exclude August because it is incomplete
full_months = monthly_recovery[
    monthly_recovery["month"] <= pd.Period("2026-07", freq="M")
].copy()

jan_amount = full_months.iloc[0]["gross_successful_amount"]
jul_amount = full_months.iloc[-1]["gross_successful_amount"]

overall_change_pct = (jul_amount - jan_amount) / jan_amount * 100

print(f"January gross successful payments: ₹{jan_amount:,.2f}")
print(f"July gross successful payments:   ₹{jul_amount:,.2f}")
print(f"January-to-July change:            {overall_change_pct:.2f}%")
print("\nMonthly changes:")
display(
    full_months[["month", "amount_crore", "mom_change_pct"]].round(2)
)

January gross successful payments: ₹187,229,127.59
July gross successful payments:   ₹187,242,265.08
January-to-July change:            0.01%

Monthly changes:


,month,amount_crore,mom_change_pct
0,2026-01,18.72,NaN
1,2026-02,17.01,-9.13
2,2026-03,18.89,11.03
3,2026-04,17.51,-7.29
4,2026-05,18.43,5.20
5,2026-06,17.56,-4.72
6,2026-07,18.72,6.65


In [66]:
# Monthly targeting volume
targeting = data["daily_targeting"].copy()

targeting["target_date"] = pd.to_datetime(
    targeting["target_date"]
)

targeting["month"] = targeting["target_date"].dt.to_period("M")

monthly_targeting = (
    targeting.groupby("month")
    .agg(
        targeting_records=("target_id", "nunique"),
        unique_accounts_targeted=("account_id", "nunique")
    )
    .reset_index()
)

# Merge with monthly payment results
activity_vs_payments = monthly_recovery.merge(
    monthly_targeting,
    on="month",
    how="left"
)

# Exclude August from this comparison because it is incomplete
activity_vs_payments = activity_vs_payments[
    activity_vs_payments["month"] <= pd.Period("2026-07", freq="M")
]

display(
    activity_vs_payments[
        [
            "month",
            "amount_crore",
            "mom_change_pct",
            "targeting_records",
            "unique_accounts_targeted"
        ]
    ].round(2)
)

,month,amount_crore,mom_change_pct,targeting_records,unique_accounts_targeted
0,2026-01,18.72,NaN,6369,5732
1,2026-02,17.01,-9.13,5709,5160
2,2026-03,18.89,11.03,6290,5666
3,2026-04,17.51,-7.29,6205,5585
4,2026-05,18.43,5.20,6442,5800
5,2026-06,17.56,-4.72,6154,5535
6,2026-07,18.72,6.65,6230,5666


In [67]:
activity_vs_payments["amount_per_target_record"] = (
    activity_vs_payments["amount_crore"] * 10_000_000
    / activity_vs_payments["targeting_records"]
)

display(
    activity_vs_payments[
        [
            "month",
            "amount_crore",
            "targeting_records",
            "amount_per_target_record"
        ]
    ].round(2)
)

,month,amount_crore,targeting_records,amount_per_target_record
0,2026-01,18.72,6369,29396.94
1,2026-02,17.01,5709,29802.50
2,2026-03,18.89,6290,30033.76
3,2026-04,17.51,6205,28225.31
4,2026-05,18.43,6442,28601.41
5,2026-06,17.56,6154,28527.74
6,2026-07,18.72,6230,30054.94


In [68]:
accounts = data["accounts"].copy()
payments = payments_consolidated.copy()

# Keep only successful payments
successful_payments = payments[
    payments["payment_status"] == "SUCCESS"
].copy()

# Aggregate successful payments by account
account_payment_summary = (
    successful_payments.groupby("account_id")
    .agg(
        successful_payment_count=("payment_id", "nunique"),
        successful_payment_amount=("amount", "sum")
    )
    .reset_index()
)

# Attach payment results to account information
account_recovery = accounts.merge(
    account_payment_summary,
    on="account_id",
    how="left"
)

account_recovery["successful_payment_count"] = (
    account_recovery["successful_payment_count"].fillna(0)
)

account_recovery["successful_payment_amount"] = (
    account_recovery["successful_payment_amount"].fillna(0)
)

# Summarize by risk segment
risk_summary = (
    account_recovery.groupby("risk_segment")
    .agg(
        accounts=("account_id", "nunique"),
        accounts_with_payments=(
            "successful_payment_count",
            lambda x: (x > 0).sum()
        ),
        outstanding_amount=("outstanding_amount", "sum"),
        successful_payment_amount=(
            "successful_payment_amount", "sum"
        )
    )
    .reset_index()
)

risk_summary["payment_account_rate_pct"] = (
    risk_summary["accounts_with_payments"]
    / risk_summary["accounts"] * 100
)

display(risk_summary.round(2))

,risk_segment,accounts,accounts_with_payments,outstanding_amount,successful_payment_amount,payment_account_rate_pct
0,HIGH,7552,3300,2.646183e+09,3.315982e+08,43.70
1,LOW,7513,3381,2.633311e+09,3.308578e+08,45.00
2,MEDIUM,7533,3335,2.628179e+09,3.305101e+08,44.27
3,NPA,7402,3268,2.581362e+09,3.226178e+08,44.15


In [69]:
risk_summary["payment_to_outstanding_pct"] = (
    risk_summary["successful_payment_amount"]
    / risk_summary["outstanding_amount"]
    * 100
)

display(
    risk_summary[
        [
            "risk_segment",
            "outstanding_amount",
            "successful_payment_amount",
            "payment_to_outstanding_pct"
        ]
    ].round(2)
)

,risk_segment,outstanding_amount,successful_payment_amount,payment_to_outstanding_pct
0,HIGH,2.646183e+09,3.315982e+08,12.53
1,LOW,2.633311e+09,3.308578e+08,12.56
2,MEDIUM,2.628179e+09,3.305101e+08,12.58
3,NPA,2.581362e+09,3.226178e+08,12.50


In [70]:
import pandas as pd

# Copy the source tables
payments = payments_consolidated.copy()
targeting = data["daily_targeting"].copy()

# Keep successful payments only
successful_payments = payments[
    payments["payment_status"] == "SUCCESS"
].copy()

# Convert timestamps
successful_payments["event_at"] = pd.to_datetime(
    successful_payments["event_at"]
)

targeting["target_date"] = pd.to_datetime(
    targeting["target_date"]
)

# Match payments to targeting records using account_id
matches = successful_payments.merge(
    targeting,
    on="account_id",
    how="inner"
)

# Keep targeting records from the 30 days before payment
matches["days_after_targeting"] = (
    matches["event_at"] - matches["target_date"]
).dt.total_seconds() / 86400

eligible_matches = matches[
    (matches["days_after_targeting"] >= 0) &
    (matches["days_after_targeting"] <= 30)
].copy()

# Attribute each payment to its most recent eligible targeting record
attributed_payments = (
    eligible_matches
    .sort_values("days_after_targeting")
    .drop_duplicates(subset="payment_id", keep="first")
)

print("Attributed payment records:", len(attributed_payments))
print(
    "Unique attributed payments:",
    attributed_payments["payment_id"].nunique()
)

Attributed payment records: 3012
Unique attributed payments: 3012


In [71]:
channel_summary = (
    attributed_payments.groupby("recommended_channel")
    .agg(
        successful_payments=("payment_id", "nunique"),
        unique_accounts=("account_id", "nunique"),
        gross_payment_amount=("amount", "sum")
    )
    .reset_index()
)

channel_summary["gross_amount_crore"] = (
    channel_summary["gross_payment_amount"] / 10_000_000
)

display(
    channel_summary[
        [
            "recommended_channel",
            "successful_payments",
            "unique_accounts",
            "gross_amount_crore"
        ]
    ].round(2)
)

,recommended_channel,successful_payments,unique_accounts,gross_amount_crore
0,FIELD,735,712,5.55
1,SMS,761,725,5.60
2,VOICE,782,750,5.99
3,WHATSAPP,734,700,5.62


In [72]:
calls = data["calls"].copy()
campaigns = data["campaigns"].copy()

# Attach campaign channel information to calls
calls_with_channel = calls.merge(
    campaigns[["campaign_id", "channel"]],
    on="campaign_id",
    how="left"
)

# Summarize call outcomes by campaign channel
call_channel_summary = (
    calls_with_channel.groupby("channel")
    .agg(
        total_calls=("call_id", "size"),
        answered_calls=(
            "call_status",
            lambda x: (x == "ANSWERED").sum()
        ),
        unique_accounts=("account_id", "nunique")
    )
    .reset_index()
)

call_channel_summary["answer_rate_pct"] = (
    call_channel_summary["answered_calls"]
    / call_channel_summary["total_calls"] * 100
)

display(call_channel_summary.round(2))

,channel,total_calls,answered_calls,unique_accounts,answer_rate_pct
0,FIELD,13639,2704,10890,19.83
1,MIXED,17326,3490,12997,20.14
2,SMS,21368,4240,15110,19.84
3,VOICE,15282,2949,11743,19.30
4,WHATSAPP,23735,4763,16262,20.07


In [73]:
# Copy account-level data
accounts = data["accounts"].copy()

# Define DPD buckets
accounts["dpd_bucket"] = pd.cut(
    accounts["dpd"],
    bins=[-1, 0, 30, 60, 90, 180],
    labels=["0 days", "1-30 days", "31-60 days", "61-90 days", "91-180 days"]
)

# Join successful payment summary to accounts
dpd_analysis = accounts.merge(
    account_payment_summary,
    on="account_id",
    how="left"
)

dpd_analysis["successful_payment_count"] = (
    dpd_analysis["successful_payment_count"].fillna(0)
)

dpd_analysis["successful_payment_amount"] = (
    dpd_analysis["successful_payment_amount"].fillna(0)
)

# Summarize by DPD bucket
dpd_summary = (
    dpd_analysis.groupby("dpd_bucket", observed=False)
    .agg(
        accounts=("account_id", "nunique"),
        accounts_with_payments=(
            "successful_payment_count",
            lambda x: (x > 0).sum()
        ),
        successful_payment_amount=(
            "successful_payment_amount", "sum"
        )
    )
    .reset_index()
)

dpd_summary["payment_account_rate_pct"] = (
    dpd_summary["accounts_with_payments"]
    / dpd_summary["accounts"] * 100
)

display(dpd_summary.round(2))

,dpd_bucket,accounts,accounts_with_payments,successful_payment_amount,payment_account_rate_pct
0,0 days,2685,1220,1.174619e+08,45.44
1,1-30 days,10880,4719,4.715398e+08,43.37
2,31-60 days,5514,2524,2.527771e+08,45.77
3,61-90 days,5468,2436,2.387357e+08,44.55
4,91-180 days,5453,2385,2.350695e+08,43.74


In [74]:
status_dpd_summary = (
    dpd_analysis.groupby(
        ["dpd_bucket", "status"],
        observed=False
    )
    .agg(
        accounts=("account_id", "nunique"),
        accounts_with_payments=(
            "successful_payment_count",
            lambda x: (x > 0).sum()
        ),
        successful_payment_amount=(
            "successful_payment_amount", "sum"
        )
    )
    .reset_index()
)

status_dpd_summary["payment_account_rate_pct"] = (
    status_dpd_summary["accounts_with_payments"]
    / status_dpd_summary["accounts"] * 100
)

display(status_dpd_summary.round(2))

,dpd_bucket,status,accounts,accounts_with_payments,successful_payment_amount,payment_account_rate_pct
0,0 days,ACTIVE,707,340,3.217378e+07,48.09
1,0 days,CLOSED,698,312,2.908689e+07,44.70
2,0 days,PAID,673,304,2.927535e+07,45.17
3,0 days,WRITEOFF,607,264,2.692591e+07,43.49
4,1-30 days,ACTIVE,2711,1171,1.187162e+08,43.19
5,1-30 days,CLOSED,2713,1181,1.198889e+08,43.53
6,1-30 days,PAID,2704,1141,1.137414e+08,42.20
7,1-30 days,WRITEOFF,2752,1226,1.191933e+08,44.55
8,31-60 days,ACTIVE,1376,608,6.317848e+07,44.19
9,31-60 days,CLOSED,1380,637,6.144156e+07,46.16


In [75]:
# Prepare successful payments
successful_payments = payments_consolidated[
    payments_consolidated["payment_status"] == "SUCCESS"
].copy()

successful_payments["event_at"] = pd.to_datetime(
    successful_payments["event_at"], errors="coerce"
)

# Prepare account status history
status_history = data["account_status_history"].copy()

status_history["event_at"] = pd.to_datetime(
    status_history["event_at"], errors="coerce"
)

# Focus on PAID and CLOSED status events
paid_closed_history = status_history[
    status_history["status"].isin(["PAID", "CLOSED"])
].copy()

# Match payments and status events by account
payment_status_timing = successful_payments[
    ["payment_id", "account_id", "event_at", "amount"]
].merge(
    paid_closed_history[
        ["account_id", "event_at", "status"]
    ],
    on="account_id",
    how="inner",
    suffixes=("_payment", "_status")
)

# Calculate timing difference in days
payment_status_timing["days_from_status_event"] = (
    payment_status_timing["event_at_payment"]
    - payment_status_timing["event_at_status"]
).dt.total_seconds() / 86400

display(
    payment_status_timing[
        [
            "payment_id",
            "account_id",
            "status",
            "event_at_payment",
            "event_at_status",
            "days_from_status_event"
        ]
    ].head(10)
)

,payment_id,account_id,status,event_at_payment,event_at_status,days_from_status_event
0,PAYMENT0000005,ACC0001770,PAID,2026-03-03 06:08:23,2026-01-15 22:52:51,46.302454
1,PAYMENT0000005,ACC0001770,CLOSED,2026-03-03 06:08:23,2026-06-25 18:10:51,-114.501713
2,PAYMENT0000008,ACC0027360,PAID,2026-01-16 08:37:42,2026-05-27 08:36:20,-130.999051
3,PAYMENT0000009,ACC0005092,PAID,2026-03-31 08:11:38,2026-07-12 14:15:34,-103.252731
4,PAYMENT0000013,ACC0026504,PAID,2026-05-06 22:30:42,2026-01-07 19:10:29,119.139039
5,PAYMENT0000014,ACC0017077,CLOSED,2026-03-08 16:59:47,2026-07-29 09:09:23,-142.673333
6,PAYMENT0000014,ACC0017077,PAID,2026-03-08 16:59:47,2026-06-30 07:11:11,-113.591250
7,PAYMENT0000018,ACC0000379,PAID,2026-06-15 08:11:44,2026-04-26 07:23:10,50.033727
8,PAYMENT0000025,ACC0016158,PAID,2026-05-23 15:23:07,2026-03-31 20:00:55,52.807083
9,PAYMENT0000025,ACC0016158,CLOSED,2026-05-23 15:23:07,2026-04-15 23:08:22,37.676910


In [76]:
# Classify payment timing relative to each status event
payment_status_timing["timing_category"] = pd.cut(
    payment_status_timing["days_from_status_event"],
    bins=[float("-inf"), -1, 1, float("inf")],
    labels=[
        "Payment before status event",
        "Within 1 day of status event",
        "Payment after status event"
    ]
)

timing_summary = (
    payment_status_timing.groupby(
        ["status", "timing_category"],
        observed=False
    )
    .agg(
        payment_rows=("payment_id", "size"),
        unique_payments=("payment_id", "nunique"),
        unique_accounts=("account_id", "nunique")
    )
    .reset_index()
)

display(timing_summary)

,status,timing_category,payment_rows,unique_payments,unique_accounts
0,CLOSED,Payment before status event,2472,2259,1864
1,CLOSED,Within 1 day of status event,49,49,49
2,CLOSED,Payment after status event,2453,2227,1852
3,PAID,Payment before status event,2546,2338,1942
4,PAID,Within 1 day of status event,44,43,43
5,PAID,Payment after status event,2516,2286,1884


In [77]:
# Sort status events chronologically
status_events = paid_closed_history[
    ["account_id", "event_at", "status"]
].copy()

status_events = status_events.rename(
    columns={
        "event_at": "status_event_at",
        "status": "status_event"
    }
)

# Match each payment to status events for the same account
payment_status_check = successful_payments[
    ["payment_id", "account_id", "event_at", "amount"]
].merge(
    status_events,
    on="account_id",
    how="left"
)

# Keep only status events on or before the payment
payment_status_check = payment_status_check[
    payment_status_check["status_event_at"]
    <= payment_status_check["event_at"]
].copy()

# Select the most recent qualifying status event per payment
payment_status_check = (
    payment_status_check
    .sort_values("status_event_at")
    .drop_duplicates("payment_id", keep="last")
)

display(payment_status_check.head(10))

print(
    "Successful payments with a prior PAID/CLOSED event:",
    payment_status_check["payment_id"].nunique()
)

print(
    "Unique accounts involved:",
    payment_status_check["account_id"].nunique()
)

,payment_id,account_id,event_at,amount,status_event_at,status_event
11564,PAYMENT0014429,ACC0019055,2026-01-22 21:44:19,16261.24,2026-01-01 00:15:52,CLOSED
6828,PAYMENT0008492,ACC0001981,2026-01-30 18:00:14,19601.39,2026-01-01 00:57:47,CLOSED
12225,PAYMENT0015271,ACC0001981,2026-05-26 14:10:59,140912.65,2026-01-01 00:57:47,CLOSED
407,PAYMENT0000500,ACC0027244,2026-07-02 00:39:39,105458.68,2026-01-01 01:19:24,CLOSED
4373,PAYMENT0005494,ACC0007687,2026-01-05 02:14:32,148646.55,2026-01-01 02:52:31,CLOSED
7623,PAYMENT0009466,ACC0018290,2026-07-20 23:24:35,86930.20,2026-01-01 02:59:01,CLOSED
116,PAYMENT0000149,ACC0018444,2026-07-25 03:04:14,96766.12,2026-01-01 03:21:21,PAID
14879,PAYMENT0018652,ACC0012641,2026-03-28 06:29:27,64717.99,2026-01-01 03:58:03,CLOSED
6465,PAYMENT0008040,ACC0028227,2026-05-16 02:27:05,72569.76,2026-01-01 07:48:50,CLOSED
13266,PAYMENT0016590,ACC0028227,2026-07-17 15:44:36,33198.67,2026-01-01 07:48:50,CLOSED


Successful payments with a prior PAID/CLOSED event: 4177
Unique accounts involved: 3433


## 9. Entity resolution and historical consistency

Check account-to-borrower mappings, duplicate borrower identities, timestamps, and account status history. Flag ambiguous mappings rather than forcing an unsupported match.

In [78]:
# Prepare all account status events
all_status_events = data["account_status_history"].copy()

all_status_events["event_at"] = pd.to_datetime(
    all_status_events["event_at"],
    errors="coerce"
)

# Count status events by status
status_event_summary = (
    all_status_events.groupby("status")
    .agg(
        total_events=("history_id", "count"),
        unique_accounts=("account_id", "nunique")
    )
    .sort_values("total_events", ascending=False)
    .reset_index()
)

display(status_event_summary)

# Inspect the complete status history of one example account
example_account = "ACC0019055"

example_history = (
    all_status_events[
        all_status_events["account_id"] == example_account
    ]
    .sort_values("event_at")
)

display(example_history)

,status,total_events,unique_accounts
0,PAID,8650,7524
1,CLOSED,8614,7469
2,DELINQUENT,8612,7457
3,NPA,8612,7480
4,WRITEOFF,8583,7508
5,ACTIVE,8518,7429
6,PTP,8411,7286


,history_id,account_id,borrower_id,event_at,status,changed_by,source,recorded_at
38248,HISTORY0038249,ACC0019055,BRW0004916,2026-01-01 00:15:52,CLOSED,AGT0000074,CORE,2025-12-31 16:43:51
37862,HISTORY0037863,ACC0019055,BRW0002529,2026-05-04 21:14:50,DELINQUENT,AGT0000065,CALL,2026-05-05 16:08:20
5796,HISTORY0005797,ACC0019055,BRW0003503,2026-05-23 16:27:42,CLOSED,AGT0000088,PAYMENT,2026-05-23 13:17:48
37205,HISTORY0037206,ACC0019055,BRW0004930,2026-05-29 21:24:44,CLOSED,AGT0000082,FIELD,2026-05-29 04:14:03


In [79]:
# Count distinct borrower IDs associated with each account
account_borrower_check = (
    all_status_events.groupby("account_id")
    .agg(
        unique_borrowers=("borrower_id", "nunique"),
        history_events=("history_id", "count")
    )
    .reset_index()
)

# Identify accounts linked to multiple borrowers
inconsistent_accounts = account_borrower_check[
    account_borrower_check["unique_borrowers"] > 1
].copy()

print("Accounts in status history:", len(account_borrower_check))
print("Accounts linked to multiple borrowers:", len(inconsistent_accounts))

print(
    "Percentage affected:",
    round(
        len(inconsistent_accounts) / len(account_borrower_check) * 100,
        2
    ),
    "%"
)

display(
    inconsistent_accounts
    .sort_values("unique_borrowers", ascending=False)
    .head(10)
)

Accounts in status history: 25999
Accounts linked to multiple borrowers: 17820
Percentage affected: 68.54 %


,account_id,unique_borrowers,history_events
7458,ACC0008596,9,9
16573,ACC0019139,9,9
16856,ACC0019461,9,9
3021,ACC0003466,9,9
22097,ACC0025503,9,9
25466,ACC0029379,8,8
25818,ACC0029801,8,8
8393,ACC0009699,8,8
16323,ACC0018850,8,8
17170,ACC0019814,8,8


In [80]:
# Get the current account-to-borrower mapping
account_mapping = data["accounts"][
    ["account_id", "borrower_id"]
].copy()

# Count distinct borrowers per account in the history
history_mapping = (
    all_status_events.groupby("account_id")
    .agg(
        history_borrower_count=("borrower_id", "nunique")
    )
    .reset_index()
)

# Compare historical and current borrower mappings
mapping_check = history_mapping.merge(
    account_mapping,
    on="account_id",
    how="left"
)

# Check whether the current borrower appears in the history
historical_borrowers = (
    all_status_events.groupby("account_id")["borrower_id"]
    .agg(set)
    .rename("historical_borrowers")
    .reset_index()
)

mapping_check = mapping_check.merge(
    historical_borrowers,
    on="account_id",
    how="left"
)

mapping_check["current_borrower_in_history"] = mapping_check.apply(
    lambda row: row["borrower_id"] in row["historical_borrowers"]
    if pd.notna(row["borrower_id"]) else False,
    axis=1
)

print("Accounts checked:", len(mapping_check))
print(
    "Current borrower missing from history:",
    (~mapping_check["current_borrower_in_history"]).sum()
)

display(
    mapping_check.sort_values(
        "history_borrower_count",
        ascending=False
    ).head(10)
)

Accounts checked: 25999
Current borrower missing from history: 25996


,account_id,history_borrower_count,borrower_id,historical_borrowers,current_borrower_in_history
16573,ACC0019139,9,BRW0006412,"{BRW0005874, BRW0004288, BRW0004505, BRW000307...",False
16856,ACC0019461,9,BRW0009427,"{BRW0009169, BRW0004550, BRW0009412, BRW000076...",False
22097,ACC0025503,9,BRW0000971,"{BRW0011817, BRW0006558, BRW0008223, BRW000208...",False
3021,ACC0003466,9,BRW0010536,"{BRW0000597, BRW0008788, BRW0002266, BRW000455...",False
7458,ACC0008596,9,BRW0006205,"{BRW0011367, BRW0001036, BRW0011077, BRW000802...",False
11234,ACC0012977,8,BRW0011125,"{BRW0003480, BRW0005307, BRW0007597, BRW000465...",False
10694,ACC0012348,8,BRW0006205,"{BRW0008171, BRW0006967, BRW0008088, BRW000494...",False
10976,ACC0012677,8,BRW0006015,"{BRW0002719, BRW0001165, BRW0002744, BRW000284...",False
21316,ACC0024595,8,BRW0003095,"{BRW0009059, BRW0006835, BRW0009629, BRW000598...",False
5936,ACC0006825,8,BRW0005269,"{BRW0002641, BRW0007002, BRW0008617, BRW000158...",False


In [81]:
# Compare borrower mappings across the main tables
tables_to_check = [
    "accounts",
    "account_status_history",
    "payments",
    "calls",
    "promises_to_pay",
    "field_visits"
]

for table_name in tables_to_check:
    df = data[table_name]

    if "account_id" in df.columns and "borrower_id" in df.columns:
        mapping = df[["account_id", "borrower_id"]].dropna()

        borrowers_per_account = (
            mapping.groupby("account_id")["borrower_id"]
            .nunique()
        )

        print(f"\n{table_name}")
        print("Rows with both IDs:", len(mapping))
        print("Accounts:", mapping["account_id"].nunique())
        print(
            "Accounts with multiple borrower IDs:",
            (borrowers_per_account > 1).sum()
        )


accounts
Rows with both IDs: 29545
Accounts: 29545
Accounts with multiple borrower IDs: 0

account_status_history
Rows with both IDs: 60000
Accounts: 25999
Accounts with multiple borrower IDs: 17820

payments
Rows with both IDs: 25500
Accounts: 16934
Accounts with multiple borrower IDs: 6114

calls
Rows with both IDs: 91350
Accounts: 28408
Accounts with multiple borrower IDs: 23990

promises_to_pay
Rows with both IDs: 18000
Accounts: 13532
Accounts with multiple borrower IDs: 3675

field_visits
Rows with both IDs: 25000
Accounts: 16908
Accounts with multiple borrower IDs: 6127


In [82]:
# Build a provisional account-to-borrower reference
reference_mapping = (
    data["accounts"]
    [["account_id", "borrower_id"]]
    .dropna(subset=["account_id"])
    .drop_duplicates(subset=["account_id"])
    .rename(columns={"borrower_id": "reference_borrower_id"})
)

tables_to_check = [
    "account_status_history",
    "payments",
    "calls",
    "promises_to_pay",
    "field_visits"
]

for table_name in tables_to_check:
    df = data[table_name][["account_id", "borrower_id"]].copy()

    checked = df.merge(
        reference_mapping,
        on="account_id",
        how="left"
    )

    checked["mapping_result"] = "MATCH"

    checked.loc[
        checked["reference_borrower_id"].isna(),
        "mapping_result"
    ] = "ACCOUNT_NOT_IN_REFERENCE"

    checked.loc[
        checked["borrower_id"].isna(),
        "mapping_result"
    ] = "MISSING_BORROWER_ID"

    checked.loc[
        checked["borrower_id"].notna()
        & checked["reference_borrower_id"].notna()
        & (
            checked["borrower_id"]
            != checked["reference_borrower_id"]
        ),
        "mapping_result"
    ] = "BORROWER_MISMATCH"

    print(f"\n{table_name}")
    display(
        checked["mapping_result"]
        .value_counts()
        .rename_axis("mapping_result")
        .reset_index(name="rows")
    )


account_status_history


,mapping_result,rows
0,BORROWER_MISMATCH,59035
1,ACCOUNT_NOT_IN_REFERENCE,962
2,MATCH,3



payments


,mapping_result,rows
0,BORROWER_MISMATCH,25113
1,ACCOUNT_NOT_IN_REFERENCE,386
2,MATCH,1



calls


,mapping_result,rows
0,BORROWER_MISMATCH,89939
1,ACCOUNT_NOT_IN_REFERENCE,1403
2,MATCH,8



promises_to_pay


,mapping_result,rows
0,BORROWER_MISMATCH,17706
1,ACCOUNT_NOT_IN_REFERENCE,293
2,MATCH,1



field_visits


,mapping_result,rows
0,BORROWER_MISMATCH,24621
1,ACCOUNT_NOT_IN_REFERENCE,377
2,MATCH,2


In [83]:
# Build the set of borrower IDs in the borrower master
borrower_master = data["borrowers"].copy()

valid_borrower_ids = set(
    borrower_master["borrower_id"].dropna().unique()
)

tables_to_check = [
    "accounts",
    "account_status_history",
    "payments",
    "calls",
    "promises_to_pay",
    "field_visits"
]

for table_name in tables_to_check:
    df = data[table_name]

    borrower_ids = df["borrower_id"].dropna()

    invalid_mask = ~borrower_ids.isin(valid_borrower_ids)

    print(f"\n{table_name}")
    print("Rows with borrower ID:", len(borrower_ids))
    print("Distinct borrower IDs:", borrower_ids.nunique())
    print("Rows with IDs absent from borrower master:", invalid_mask.sum())
    print(
        "Percentage absent:",
        round(invalid_mask.mean() * 100, 2) if len(invalid_mask) else 0,
        "%"
    )


accounts
Rows with borrower ID: 29545
Distinct borrower IDs: 10943
Rows with IDs absent from borrower master: 2458
Percentage absent: 8.32 %

account_status_history
Rows with borrower ID: 60000
Distinct borrower IDs: 11916
Rows with IDs absent from borrower master: 5013
Percentage absent: 8.36 %

payments
Rows with borrower ID: 25500
Distinct borrower IDs: 10474
Rows with IDs absent from borrower master: 2078
Percentage absent: 8.15 %

calls
Rows with borrower ID: 91350
Distinct borrower IDs: 11992
Rows with IDs absent from borrower master: 7451
Percentage absent: 8.16 %

promises_to_pay
Rows with borrower ID: 18000
Distinct borrower IDs: 9299
Rows with IDs absent from borrower master: 1444
Percentage absent: 8.02 %

field_visits
Rows with borrower ID: 25000
Distinct borrower IDs: 10537
Rows with IDs absent from borrower master: 2095
Percentage absent: 8.38 %


In [84]:
# Check borrower IDs absent from the borrower master
valid_borrower_ids = set(
    data["borrowers"]["borrower_id"].dropna().unique()
)

for table_name in [
    "accounts",
    "account_status_history",
    "payments",
    "calls",
    "promises_to_pay",
    "field_visits"
]:
    df = data[table_name].copy()

    invalid_rows = df[
        df["borrower_id"].notna()
        & ~df["borrower_id"].isin(valid_borrower_ids)
    ]

    print(f"\n{table_name}")
    print("Rows with missing-master borrower IDs:", len(invalid_rows))

    if "account_id" in invalid_rows.columns:
        print(
            "Distinct affected accounts:",
            invalid_rows["account_id"].nunique()
        )

        display(
            invalid_rows[
                ["account_id", "borrower_id"]
            ]
            .drop_duplicates()
            .head(5)
        )


accounts
Rows with missing-master borrower IDs: 2458
Distinct affected accounts: 2458


,account_id,borrower_id
7,ACC0000008,BRW0011565
42,ACC0000043,BRW0000499
48,ACC0000049,BRW0010045
68,ACC0000069,BRW0001302
74,ACC0000075,BRW0004666



account_status_history
Rows with missing-master borrower IDs: 5013
Distinct affected accounts: 4618


,account_id,borrower_id
4,ACC0022967,BRW0009477
21,ACC0017419,BRW0003350
22,ACC0010185,BRW0009812
25,ACC0029652,BRW0003784
70,ACC0012669,BRW0001335



payments
Rows with missing-master borrower IDs: 2078
Distinct affected accounts: 1976


...

,account_id,borrower_id
21,ACC0009881,BRW0008929
22,ACC0005656,BRW0006248
32,ACC0020302,BRW0010171
34,ACC0024768,BRW0007892
45,ACC0008985,BRW0006195



promises_to_pay
Rows with missing-master borrower IDs: 1444
Distinct affected accounts: 1404


,account_id,borrower_id
5,ACC0020223,BRW0004429
22,ACC0015191,BRW0004509
28,ACC0002944,BRW0009884
31,ACC0021064,BRW0008881
46,ACC0010782,BRW0001525



field_visits
Rows with missing-master borrower IDs: 2095
Distinct affected accounts: 2027


,account_id,borrower_id
3,ACC0029193,BRW0005195
31,ACC0005778,BRW0011300
32,ACC0009770,BRW0001169
47,ACC0027732,BRW0008785
54,ACC0024251,BRW0004914


In [85]:
borrowers = data["borrowers"].copy()

print("Total borrower rows:", len(borrowers))
print("Unique borrower IDs:", borrowers["borrower_id"].nunique())
print(
    "Rows beyond one record per borrower ID:",
    borrowers["borrower_id"].duplicated().sum()
)

# Inspect repeated borrower IDs
repeated_borrowers = borrowers[
    borrowers["borrower_id"].duplicated(keep=False)
].sort_values("borrower_id")

display(repeated_borrowers.head(15))

# Count records per borrower ID
borrower_record_counts = (
    borrowers.groupby("borrower_id")
    .size()
    .value_counts()
    .sort_index()
    .rename_axis("records_per_borrower")
    .reset_index(name="number_of_borrowers")
)

display(borrower_record_counts)

Total borrower rows: 30600
Unique borrower IDs: 11015
Rows beyond one record per borrower ID: 19585


,borrower_id,name,phone,email,city,created_at,updated_at,state
5082,BRW0000001,Aarav Sharma,9.962893e+09,user8164@example.com,Kolkata,2025-11-04 23:36:22,2026-06-22 12:05:30,West Bengal
27686,BRW0000001,Aarav Sharma,9.237818e+09,user10670@example.com,Hyderabad,2025-12-03 00:18:45,2026-06-25 02:44:50,Telangana
20435,BRW0000001,Rohan Patel,9.712907e+09,user4799@example.com,Hyderabad,2026-07-17 04:33:13,2025-11-22 01:34:14,Telangana
20050,BRW0000004,Vikram Shah,9.072501e+09,user13710@example.com,Gurugram,2025-11-16 08:32:03,2025-10-31 11:10:43,Haryana
23818,BRW0000004,Ananya Rao,9.076787e+09,user15119@example.com,Delhi,2025-07-20 23:45:10,2025-12-20 18:43:35,Delhi
10963,BRW0000004,Neha Singh,9.714461e+09,user18767@example.com,Bengaluru,2025-07-03 17:14:49,2025-01-15 08:16:46,Karnataka
29546,BRW0000004,Pooja Nair,9.936985e+09,user11254@example.com,Mumbai,2026-02-10 12:40:50,2025-12-16 20:36:28,Maharashtra
30325,BRW0000005,Neha Singh,9.269813e+09,user4520@example.com,Mumbai,2026-02-07 14:08:27,2025-12-04 08:52:09,Maharashtra
18692,BRW0000005,Neha Singh,9.269813e+09,user4520@example.com,Mumbai,2026-02-07 14:08:27,2025-12-04 08:52:09,Maharashtra
8282,BRW0000005,Sneha Das,9.359023e+09,user5979@example.com,Pune,2026-03-18 14:01:59,2025-07-07 09:59:12,Maharashtra


,records_per_borrower,number_of_borrowers
0,1,2449
1,2,3023
2,3,2449
3,4,1645
4,5,840
5,6,395
6,7,134
7,8,57
8,9,18
9,10,3


In [86]:
borrowers = data["borrowers"].copy()

# Count distinct complete profiles for each borrower ID
profile_columns = [
    col for col in borrowers.columns
    if col != "borrower_id"
]

profile_counts = (
    borrowers.groupby("borrower_id")[profile_columns]
    .apply(lambda group: group.drop_duplicates().shape[0])
    .rename("distinct_profiles")
    .reset_index()
)

# Classify borrower IDs
profile_counts["profile_type"] = profile_counts[
    "distinct_profiles"
].map(
    lambda n: "Single profile" if n == 1
    else "Conflicting profiles"
)

print("Borrower IDs by profile consistency:")

display(
    profile_counts["profile_type"]
    .value_counts()
    .rename_axis("profile_type")
    .reset_index(name="borrower_ids")
)

print("\nBorrower IDs with the most conflicting profiles:")

display(
    profile_counts.sort_values(
        "distinct_profiles",
        ascending=False
    ).head(10)
)

Borrower IDs by profile consistency:


,profile_type,borrower_ids
0,Conflicting profiles,8518
1,Single profile,2497



Borrower IDs with the most conflicting profiles:


,borrower_id,distinct_profiles,profile_type
8150,BRW0008879,11,Conflicting profiles
2327,BRW0002520,10,Conflicting profiles
4966,BRW0005385,10,Conflicting profiles
1153,BRW0001237,10,Conflicting profiles
5812,BRW0006302,10,Conflicting profiles
6461,BRW0007009,9,Conflicting profiles
2669,BRW0002892,9,Conflicting profiles
4624,BRW0005021,9,Conflicting profiles
10144,BRW0011052,9,Conflicting profiles
1863,BRW0002013,9,Conflicting profiles


In [87]:
borrowers = data["borrowers"].copy()

borrowers["created_at"] = pd.to_datetime(
    borrowers["created_at"], errors="coerce"
)

borrowers["updated_at"] = pd.to_datetime(
    borrowers["updated_at"], errors="coerce"
)

# Identify records where updated_at precedes created_at
invalid_timestamp_order = borrowers[
    borrowers["created_at"].notna()
    & borrowers["updated_at"].notna()
    & (borrowers["updated_at"] < borrowers["created_at"])
]

print("Total borrower records:", len(borrowers))
print(
    "Records where updated_at precedes created_at:",
    len(invalid_timestamp_order)
)

print(
    "Percentage affected:",
    round(len(invalid_timestamp_order) / len(borrowers) * 100, 2),
    "%"
)

display(invalid_timestamp_order.head(10))

Total borrower records: 30600
Records where updated_at precedes created_at: 15354
Percentage affected: 50.18 %


,borrower_id,name,phone,email,city,created_at,updated_at,state
1,BRW0009288,Neha Singh,9.154913e+09,user9581@example.com,Gurugram,2025-06-29 04:32:01,2025-03-04 12:15:30,Haryana
2,BRW0007855,Vikram Shah,9.986755e+09,user19407@example.com,Hyderabad,2025-11-23 20:30:50,2025-01-07 21:07:17,Telangana
3,BRW0005267,Priya Mehta,9.145160e+09,user4619@example.com,Jaipur,2026-05-09 11:34:12,2025-04-24 21:35:48,Rajasthan
7,BRW0008369,Vikram Shah,9.737942e+09,user8941@example.com,Bhubaneswar,2026-05-27 08:50:59,2025-09-20 02:27:08,Odisha
8,BRW0002418,Sneha Das,9.608424e+09,user16450@example.com,Mumbai,2026-05-27 06:05:11,2025-07-19 15:06:52,Maharashtra
9,BRW0001131,Amit Kumar,9.786301e+09,user5173@example.com,Jaipur,2025-11-23 01:27:02,2025-03-21 22:18:48,Rajasthan
11,BRW0011708,Priya Mehta,9.895982e+09,user5617@example.com,Chennai,2026-04-08 06:45:08,2025-02-07 01:14:56,Tamil Nadu
13,BRW0009134,Neha Singh,9.223954e+09,user9831@example.com,Bhubaneswar,2026-02-10 17:43:22,2025-01-19 21:07:39,Odisha
14,BRW0008610,Rahul Verma,9.639690e+09,user6535@example.com,Gurugram,2026-03-29 04:42:45,2025-07-11 05:04:23,Haryana
16,BRW0006159,Rohan Patel,9.911125e+09,user8995@example.com,Kolkata,2025-11-25 04:20:45,2025-11-24 18:11:12,West Bengal


In [88]:
for table_name, df in data.items():
    if "created_at" in df.columns and "updated_at" in df.columns:
        print(table_name, "->", len(df), "rows")

borrowers -> 30600 rows


In [89]:
borrower_quality_summary = pd.DataFrame([
    {
        "issue": "Repeated borrower records",
        "finding": "30,600 rows but only 11,015 unique borrower IDs",
        "severity": "High"
    },
    {
        "issue": "Conflicting borrower profiles",
        "finding": "8,518 borrower IDs have multiple distinct profiles",
        "severity": "High"
    },
    {
        "issue": "Invalid timestamp order",
        "finding": "15,354 records (50.18%) have updated_at earlier than created_at",
        "severity": "High"
    },
    {
        "issue": "Borrower IDs missing from master",
        "finding": "Approximately 8% of borrower-ID-bearing rows in checked tables reference IDs absent from borrowers",
        "severity": "High"
    },
    {
        "issue": "Cross-table borrower mapping",
        "finding": "Operational tables frequently disagree with the provisional mapping in accounts",
        "severity": "Critical"
    }
])

display(borrower_quality_summary)

,issue,finding,severity
0,Repeated borrower records,"30,600 rows but only 11,015 unique borrower IDs",High
1,Conflicting borrower profiles,"8,518 borrower IDs have multiple distinct prof...",High
2,Invalid timestamp order,"15,354 records (50.18%) have updated_at earlie...",High
3,Borrower IDs missing from master,Approximately 8% of borrower-ID-bearing rows i...,High
4,Cross-table borrower mapping,Operational tables frequently disagree with th...,Critical


## 10. Cross-table cleaning validation

Compare duplicate patterns across tables and validate the resulting working copies. Cleaning decisions should be explainable and their impact quantified.

In [90]:
# Inspect duplicate records across all source tables

duplicate_summary = []

for table_name, df in data.items():
    duplicate_summary.append({
        "table": table_name,
        "total_rows": len(df),
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "unique_rows": len(df.drop_duplicates()),
        "columns": len(df.columns)
    })

duplicate_summary = pd.DataFrame(duplicate_summary)

display(
    duplicate_summary.sort_values(
        "exact_duplicate_rows",
        ascending=False
    )
)

,table,total_rows,exact_duplicate_rows,unique_rows,columns
7,calls,91350,1271,90079,11
4,borrowers,30600,600,30000,8
17,whatsapp_events,60600,600,60000,8
13,payments,25500,486,25014,9
1,accounts,30000,0,30000,12
0,account_status_history,60000,0,60000,8
5,call_attempts,120000,0,120000,9
6,call_dispositions,35000,0,35000,8
3,agents,30000,0,30000,8
2,agent_sessions,15000,0,15000,7


In [91]:
# Create cleaned copies by removing exact duplicate rows only

cleaned_data = {}

for table_name, df in data.items():
    cleaned_df = df.drop_duplicates().copy()
    cleaned_data[table_name] = cleaned_df

    removed = len(df) - len(cleaned_df)

    if removed > 0:
        print(
            f"{table_name}: "
            f"{removed:,} exact duplicate rows removed"
        )

print("\nCleaning complete.")

print(
    "Original tables:",
    len(data),
    "| Cleaned tables:",
    len(cleaned_data)
)

borrowers: 600 exact duplicate rows removed
calls: 1,271 exact duplicate rows removed
payments: 486 exact duplicate rows removed
whatsapp_events: 600 exact duplicate rows removed

Cleaning complete.
Original tables: 18 | Cleaned tables: 18


In [92]:
# Check key uniqueness in cleaned tables

primary_keys = {
    "accounts": "account_id",
    "account_status_history": "history_id",
    "agent_sessions": "session_id",
    "agents": "agent_id",
    "borrowers": "borrower_id",
    "call_attempts": "attempt_id",
    "call_dispositions": "disposition_id",
    "calls": "call_id",
    "campaigns": "campaign_id",
    "complaints": "complaint_id",
    "daily_targeting": "target_id",
    "field_visits": "visit_id",
    "payments": "payment_id",
    "promises_to_pay": "ptp_id",
    "sms_events": "sms_event_id",
    "vendor_telephony": "vendor_id",
    "whatsapp_events": "whatsapp_event_id"
}

key_check = []

for table_name, key in primary_keys.items():
    df = cleaned_data[table_name]

    if key not in df.columns:
        key_check.append({
            "table": table_name,
            "key": key,
            "duplicate_key_rows": None,
            "missing_key_rows": None,
            "result": "KEY COLUMN NOT FOUND"
        })
        continue

    key_check.append({
        "table": table_name,
        "key": key,
        "duplicate_key_rows": int(
            df[key].duplicated(keep=False).sum()
        ),
        "missing_key_rows": int(df[key].isna().sum()),
        "result": "CHECK REQUIRED"
    })

key_check = pd.DataFrame(key_check)
display(key_check)

,table,key,duplicate_key_rows,missing_key_rows,result
0,accounts,account_id,0,0,CHECK REQUIRED
1,account_status_history,history_id,0,0,CHECK REQUIRED
2,agent_sessions,session_id,0,0,CHECK REQUIRED
3,agents,agent_id,30000,0,CHECK REQUIRED
4,borrowers,borrower_id,27503,0,CHECK REQUIRED
5,call_attempts,attempt_id,0,0,CHECK REQUIRED
6,call_dispositions,disposition_id,0,0,CHECK REQUIRED
7,calls,call_id,158,0,CHECK REQUIRED
8,campaigns,campaign_id,0,0,CHECK REQUIRED
9,complaints,complaint_id,0,0,CHECK REQUIRED


In [93]:
# Inspect payment IDs that still appear more than once

payments_clean = cleaned_data["payments"]

duplicate_payment_ids = (
    payments_clean[
        payments_clean["payment_id"].duplicated(keep=False)
    ]
    .sort_values("payment_id")
)

print("Rows with repeated payment IDs:", len(duplicate_payment_ids))
print(
    "Number of repeated payment IDs:",
    duplicate_payment_ids["payment_id"].nunique()
)

display(duplicate_payment_ids.head(20))

Rows with repeated payment IDs: 28
Number of repeated payment IDs: 14


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
1389,PAYMENT0001390,ACC0001624,BRW0004200,2026-05-14 07:52:59,TXN0000046535,85205.16,SUCCESS,CASH,VND0000015
25043,PAYMENT0001390,ACC0001624,BRW0004200,2026-05-14 07:52:59,NaN,85205.16,SUCCESS,CASH,VND0000015
2097,PAYMENT0002098,ACC0023839,BRW0008653,2026-07-01 09:19:16,TXN0000001302,5570.92,SUCCESS,NETBANKING,VND0000001
25055,PAYMENT0002098,ACC0023839,BRW0008653,2026-07-01 09:19:16,NaN,5570.92,SUCCESS,NETBANKING,VND0000001
2302,PAYMENT0002303,ACC0021362,BRW0001779,2026-06-15 03:09:47,TXN0000067311,59429.86,SUCCESS,CARD,VND0000011
25057,PAYMENT0002303,ACC0021362,BRW0001779,2026-06-15 03:09:47,NaN,59429.86,SUCCESS,CARD,VND0000011
25437,PAYMENT0005191,ACC0001901,BRW0001234,2026-06-28 13:50:03,TXN0000041890,107235.47,SUCCESS,NACH,VND0000002
5190,PAYMENT0005191,ACC0001901,BRW0001234,2026-06-28 13:50:03,NaN,107235.47,SUCCESS,NACH,VND0000002
25385,PAYMENT0007582,ACC0017962,BRW0008041,2026-03-26 12:49:42,TXN0000013106,119714.20,SUCCESS,NETBANKING,VND0000014
7581,PAYMENT0007582,ACC0017962,BRW0008041,2026-03-26 12:49:42,NaN,119714.20,SUCCESS,NETBANKING,VND0000014


In [94]:
# Compare repeated payment IDs across important fields

duplicate_payment_ids = (
    cleaned_data["payments"][
        cleaned_data["payments"]["payment_id"].duplicated(keep=False)
    ]
    .sort_values("payment_id")
)

comparison_columns = [
    "account_id",
    "borrower_id",
    "event_at",
    "amount",
    "payment_status",
    "payment_method",
    "provider_id"
]

payment_conflicts = (
    duplicate_payment_ids
    .groupby("payment_id")[comparison_columns]
    .nunique(dropna=False)
)

print("Repeated payment IDs:", len(payment_conflicts))
print("\nIDs with conflicting core fields:")
display(payment_conflicts[(payment_conflicts > 1).any(axis=1)])

Repeated payment IDs: 14

IDs with conflicting core fields:


,account_id,borrower_id,event_at,amount,payment_status,payment_method,provider_id
payment_id,,,,,,,


In [95]:
# Consolidate payment IDs after validating core fields

payments_before = cleaned_data["payments"].copy()

core_columns = [
    "account_id",
    "borrower_id",
    "event_at",
    "amount",
    "payment_status",
    "payment_method",
    "provider_id"
]

# Confirm core fields agree within every payment ID
core_conflicts = (
    payments_before.groupby("payment_id")[core_columns]
    .nunique(dropna=False)
)

conflicting_ids = core_conflicts[
    (core_conflicts > 1).any(axis=1)
].index

if len(conflicting_ids) > 0:
    print("STOP: Conflicting payment IDs found:", list(conflicting_ids))
else:
    # Preserve the first non-missing value for each column
    payments_consolidated = (
        payments_before
        .groupby("payment_id", as_index=False, sort=False)
        .first()
    )

    print("Rows before consolidation:", len(payments_before))
    print("Rows after consolidation:", len(payments_consolidated))
    print(
        "Repeated-ID rows consolidated:",
        len(payments_before) - len(payments_consolidated)
    )
    print(
        "Unique payment IDs:",
        payments_consolidated["payment_id"].nunique()
    )

Rows before consolidation: 25014
Rows after consolidation: 25000
Repeated-ID rows consolidated: 14
Unique payment IDs: 25000


In [96]:
# Inspect repeated call IDs

calls_clean = cleaned_data["calls"]

duplicate_calls = (
    calls_clean[
        calls_clean["call_id"].duplicated(keep=False)
    ]
    .sort_values("call_id")
)

print("Rows with repeated call IDs:", len(duplicate_calls))
print(
    "Number of repeated call IDs:",
    duplicate_calls["call_id"].nunique()
)

display(duplicate_calls.head(20))

Rows with repeated call IDs: 158
Number of repeated call IDs: 79


,call_id,account_id,borrower_id,event_at,agent_id,campaign_id,direction,vendor_id,call_status,duration_sec,timezone
225,CALL0000226,ACC0010288,BRW0011653,2026-01-09 11:36:06,AGT0000584,CMP0000108,OUTBOUND,VND0000015,VOICEMAIL,184,Asia/Dubai
90485,CALL0000226,ACC0010288,BRW0011653,2026-01-12 11:36:06,AGT0000584,CMP0000108,OUTBOUND,VND0000015,VOICEMAIL,184,Asia/Dubai
1605,CALL0001606,ACC0028730,BRW0011773,2026-02-28 06:39:17,NaN,CMP0000019,OUTBOUND,VND0000007,FAILED,845,Asia/Dubai
90739,CALL0001606,ACC0028730,BRW0011773,2026-02-28 06:39:17,AGT0000504,CMP0000019,OUTBOUND,VND0000007,FAILED,845,Asia/Dubai
1810,CALL0001811,ACC0020868,BRW0000316,2026-05-14 02:18:37,NaN,CMP0000009,OUTBOUND,VND0000015,BUSY,360,Asia/Kolkata
90953,CALL0001811,ACC0020868,BRW0000316,2026-05-14 02:18:37,AGT0000531,CMP0000009,OUTBOUND,VND0000015,BUSY,360,Asia/Kolkata
90125,CALL0003082,ACC0006316,BRW0007107,2026-04-18 14:14:49,AGT0000120,CMP0000094,OUTBOUND,VND0000003,BUSY,10,Asia/Dubai
3081,CALL0003082,ACC0006316,BRW0007107,2026-04-18 14:14:49,NaN,CMP0000094,OUTBOUND,VND0000003,BUSY,10,Asia/Dubai
91153,CALL0004229,ACC0014927,BRW0000039,2026-05-16 07:06:49,AGT0000433,CMP0000082,OUTBOUND,VND0000001,NO_ANSWER,278,UTC
4228,CALL0004229,ACC0014927,BRW0000039,2026-05-16 07:06:49,NaN,CMP0000082,OUTBOUND,VND0000001,NO_ANSWER,278,UTC


In [97]:
# Check repeated call IDs for conflicting fields

call_core_columns = [
    "account_id",
    "borrower_id",
    "event_at",
    "campaign_id",
    "direction",
    "vendor_id",
    "call_status",
    "duration_sec",
    "timezone"
]

call_conflicts = (
    duplicate_calls
    .groupby("call_id")[call_core_columns]
    .nunique(dropna=False)
)

conflicting_calls = call_conflicts[
    (call_conflicts > 1).any(axis=1)
]

print("Repeated call IDs:", len(call_conflicts))
print("IDs with conflicting core fields:", len(conflicting_calls))

display(conflicting_calls.head(20))

Repeated call IDs: 79
IDs with conflicting core fields: 11


,account_id,borrower_id,event_at,campaign_id,direction,vendor_id,call_status,duration_sec,timezone
call_id,,,,,,,,,
CALL0000226,1,1,2,1,1,1,1,1,1
CALL0005099,1,1,2,1,1,1,1,1,1
CALL0008648,1,1,2,1,1,1,1,1,1
CALL0013470,1,1,2,1,1,1,1,1,1
CALL0056697,1,1,2,1,1,1,1,1,1
CALL0059442,1,1,2,1,1,1,1,1,1
CALL0063660,1,1,2,1,1,1,1,1,1
CALL0076303,1,1,2,1,1,1,1,1,1
CALL0077078,1,1,2,1,1,1,1,1,1


In [98]:
# Display every record belonging to the 11 conflicting call IDs

conflicting_call_ids = conflicting_calls.index

call_conflict_details = (
    duplicate_calls[
        duplicate_calls["call_id"].isin(conflicting_call_ids)
    ]
    .sort_values(["call_id", "event_at"])
)

display(call_conflict_details)

,call_id,account_id,borrower_id,event_at,agent_id,campaign_id,direction,vendor_id,call_status,duration_sec,timezone
225,CALL0000226,ACC0010288,BRW0011653,2026-01-09 11:36:06,AGT0000584,CMP0000108,OUTBOUND,VND0000015,VOICEMAIL,184,Asia/Dubai
90485,CALL0000226,ACC0010288,BRW0011653,2026-01-12 11:36:06,AGT0000584,CMP0000108,OUTBOUND,VND0000015,VOICEMAIL,184,Asia/Dubai
90925,CALL0005099,ACC0013498,BRW0005794,2026-07-06 10:21:04,AGT0000013,CMP0000068,INBOUND,VND0000011,NO_ANSWER,823,Asia/Kolkata
5098,CALL0005099,ACC0013498,BRW0005794,2026-07-10 10:21:04,AGT0000013,CMP0000068,INBOUND,VND0000011,NO_ANSWER,823,Asia/Kolkata
8647,CALL0008648,ACC0016926,BRW0001908,2026-06-04 21:34:24,AGT0000932,CMP0000101,OUTBOUND,VND0000002,ANSWERED,812,UTC
90332,CALL0008648,ACC0016926,BRW0001908,2026-06-07 21:34:24,AGT0000932,CMP0000101,OUTBOUND,VND0000002,ANSWERED,812,UTC
91341,CALL0013470,ACC0011890,BRW0002352,2026-03-29 06:06:00,AGT0000310,CMP0000085,OUTBOUND,VND0000005,BUSY,178,Asia/Kolkata
13469,CALL0013470,ACC0011890,BRW0002352,2026-03-31 06:06:00,AGT0000310,CMP0000085,OUTBOUND,VND0000005,BUSY,178,Asia/Kolkata
90396,CALL0056697,ACC0024264,BRW0003381,2026-02-26 00:25:38,AGT0000632,CMP0000008,OUTBOUND,VND0000013,VOICEMAIL,489,UTC
56696,CALL0056697,ACC0024264,BRW0003381,2026-03-03 00:25:38,AGT0000632,CMP0000008,OUTBOUND,VND0000013,VOICEMAIL,489,UTC


In [99]:
# Compare repeated calls after excluding event_at

call_comparison_columns = [
    col for col in duplicate_calls.columns
    if col not in ["call_id", "event_at"]
]

comparison = (
    duplicate_calls
    .groupby("call_id")[call_comparison_columns]
    .nunique(dropna=False)
)

print(
    "Repeated call IDs with differences beyond event_at:",
    int((comparison > 1).any(axis=1).sum())
)

print(
    "Repeated call IDs with no differences beyond event_at:",
    int((comparison == 1).all(axis=1).sum())
)

Repeated call IDs with differences beyond event_at: 68
Repeated call IDs with no differences beyond event_at: 11


In [100]:
# Identify the fields that differ for each repeated call ID

call_conflict_summary = []

for call_id, group in duplicate_calls.groupby("call_id"):
    differing_fields = [
        col for col in group.columns
        if group[col].nunique(dropna=False) > 1
    ]

    call_conflict_summary.append({
        "call_id": call_id,
        "rows": len(group),
        "differing_fields": ", ".join(differing_fields)
    })

call_conflict_summary = pd.DataFrame(call_conflict_summary)

display(
    call_conflict_summary
    .sort_values("differing_fields")
    .reset_index(drop=True)
)

,call_id,rows,differing_fields
0,CALL0045244,2,agent_id
1,CALL0060871,2,agent_id
2,CALL0056387,2,agent_id
3,CALL0055570,2,agent_id
4,CALL0054426,2,agent_id
...,...,...,...
74,CALL0013470,2,event_at
75,CALL0059442,2,event_at
76,CALL0056697,2,event_at
77,CALL0079063,2,event_at


In [101]:
# Count how often each field differs across repeated call IDs

field_difference_counts = {}

for call_id, group in duplicate_calls.groupby("call_id"):
    for col in group.columns:
        if group[col].nunique(dropna=False) > 1:
            field_difference_counts[col] = (
                field_difference_counts.get(col, 0) + 1
            )

field_difference_summary = (
    pd.DataFrame(
        list(field_difference_counts.items()),
        columns=["field", "repeated_ids_with_differences"]
    )
    .sort_values(
        "repeated_ids_with_differences",
        ascending=False
    )
    .reset_index(drop=True)
)

display(field_difference_summary)

,field,repeated_ids_with_differences
0,agent_id,68
1,event_at,11


In [102]:
# Separate agent_id differences from timestamp conflicts

agent_id_conflict_ids = call_conflict_summary.loc[
    call_conflict_summary["differing_fields"] == "agent_id",
    "call_id"
]

agent_id_cases = (
    duplicate_calls[
        duplicate_calls["call_id"].isin(agent_id_conflict_ids)
    ]
    .sort_values("call_id")
)

print("Repeated IDs differing only in agent_id:",
      len(agent_id_conflict_ids))

display(agent_id_cases.head(20))

Repeated IDs differing only in agent_id: 68


,call_id,account_id,borrower_id,event_at,agent_id,campaign_id,direction,vendor_id,call_status,duration_sec,timezone
1605,CALL0001606,ACC0028730,BRW0011773,2026-02-28 06:39:17,NaN,CMP0000019,OUTBOUND,VND0000007,FAILED,845,Asia/Dubai
90739,CALL0001606,ACC0028730,BRW0011773,2026-02-28 06:39:17,AGT0000504,CMP0000019,OUTBOUND,VND0000007,FAILED,845,Asia/Dubai
1810,CALL0001811,ACC0020868,BRW0000316,2026-05-14 02:18:37,NaN,CMP0000009,OUTBOUND,VND0000015,BUSY,360,Asia/Kolkata
90953,CALL0001811,ACC0020868,BRW0000316,2026-05-14 02:18:37,AGT0000531,CMP0000009,OUTBOUND,VND0000015,BUSY,360,Asia/Kolkata
90125,CALL0003082,ACC0006316,BRW0007107,2026-04-18 14:14:49,AGT0000120,CMP0000094,OUTBOUND,VND0000003,BUSY,10,Asia/Dubai
3081,CALL0003082,ACC0006316,BRW0007107,2026-04-18 14:14:49,NaN,CMP0000094,OUTBOUND,VND0000003,BUSY,10,Asia/Dubai
91153,CALL0004229,ACC0014927,BRW0000039,2026-05-16 07:06:49,AGT0000433,CMP0000082,OUTBOUND,VND0000001,NO_ANSWER,278,UTC
4228,CALL0004229,ACC0014927,BRW0000039,2026-05-16 07:06:49,NaN,CMP0000082,OUTBOUND,VND0000001,NO_ANSWER,278,UTC
7214,CALL0007215,ACC0002256,BRW0004867,2026-04-22 02:23:35,NaN,CMP0000063,OUTBOUND,VND0000010,VOICEMAIL,237,Asia/Kolkata
90302,CALL0007215,ACC0002256,BRW0004867,2026-04-22 02:23:35,AGT0000859,CMP0000063,OUTBOUND,VND0000010,VOICEMAIL,237,Asia/Kolkata


In [103]:
# Start with the cleaned calls table
calls_before = cleaned_data["calls"].copy()

# Separate the 68 agent_id cases from the 11 timestamp conflicts
agent_case_ids = call_conflict_summary.loc[
    call_conflict_summary["differing_fields"] == "agent_id",
    "call_id"
]

agent_cases = calls_before[
    calls_before["call_id"].isin(agent_case_ids)
].copy()

# Verify that each group has exactly one non-missing agent ID
agent_check = (
    agent_cases.groupby("call_id")["agent_id"]
    .nunique()
)

if (agent_check != 1).any():
    print("STOP: Some call IDs have conflicting agent IDs.")
else:
    # Fill missing agent IDs using the value within the same call ID
    agent_cases["agent_id"] = (
        agent_cases.groupby("call_id")["agent_id"]
        .transform(lambda s: s.ffill().bfill())
    )

    # Keep one row per call ID for these verified groups
    agent_cases = agent_cases.drop_duplicates(subset="call_id")

    # Exclude only these verified groups from the original table
    remaining_calls = calls_before[
        ~calls_before["call_id"].isin(agent_case_ids)
    ].copy()

    # Combine consolidated groups with all remaining records
    calls_consolidated = pd.concat(
        [remaining_calls, agent_cases],
        ignore_index=True
    )

    print("Rows before consolidation:", len(calls_before))
    print("Rows after consolidation:", len(calls_consolidated))
    print(
        "Unique call IDs:",
        calls_consolidated["call_id"].nunique()
    )
    print(
        "Timestamp-conflict groups preserved:",
        len(conflicting_calls)
        - len(agent_case_ids)
    )

Rows before consolidation: 90079
Rows after consolidation: 90011
Unique call IDs: 90000
Timestamp-conflict groups preserved: -57


In [104]:
# Validate the consolidated calls table

print("Total rows:", len(calls_consolidated))
print("Unique call IDs:", calls_consolidated["call_id"].nunique())
print("Missing call IDs:", calls_consolidated["call_id"].isna().sum())

remaining_duplicates = calls_consolidated[
    calls_consolidated["call_id"].duplicated(keep=False)
].sort_values("call_id")

print("Rows with repeated call IDs:", len(remaining_duplicates))
print("Number of repeated call IDs:", remaining_duplicates["call_id"].nunique())

display(remaining_duplicates.head(30))

Total rows: 90011
Unique call IDs: 90000
Missing call IDs: 0
Rows with repeated call IDs: 22
Number of repeated call IDs: 11


,call_id,account_id,borrower_id,event_at,agent_id,campaign_id,direction,vendor_id,call_status,duration_sec,timezone
225,CALL0000226,ACC0010288,BRW0011653,2026-01-09 11:36:06,AGT0000584,CMP0000108,OUTBOUND,VND0000015,VOICEMAIL,184,Asia/Dubai
89935,CALL0000226,ACC0010288,BRW0011653,2026-01-12 11:36:06,AGT0000584,CMP0000108,OUTBOUND,VND0000015,VOICEMAIL,184,Asia/Dubai
5094,CALL0005099,ACC0013498,BRW0005794,2026-07-10 10:21:04,AGT0000013,CMP0000068,INBOUND,VND0000011,NO_ANSWER,823,Asia/Kolkata
89939,CALL0005099,ACC0013498,BRW0005794,2026-07-06 10:21:04,AGT0000013,CMP0000068,INBOUND,VND0000011,NO_ANSWER,823,Asia/Kolkata
8641,CALL0008648,ACC0016926,BRW0001908,2026-06-04 21:34:24,AGT0000932,CMP0000101,OUTBOUND,VND0000002,ANSWERED,812,UTC
89933,CALL0008648,ACC0016926,BRW0001908,2026-06-07 21:34:24,AGT0000932,CMP0000101,OUTBOUND,VND0000002,ANSWERED,812,UTC
89942,CALL0013470,ACC0011890,BRW0002352,2026-03-29 06:06:00,AGT0000310,CMP0000085,OUTBOUND,VND0000005,BUSY,178,Asia/Kolkata
13461,CALL0013470,ACC0011890,BRW0002352,2026-03-31 06:06:00,AGT0000310,CMP0000085,OUTBOUND,VND0000005,BUSY,178,Asia/Kolkata
89934,CALL0056697,ACC0024264,BRW0003381,2026-02-26 00:25:38,AGT0000632,CMP0000008,OUTBOUND,VND0000013,VOICEMAIL,489,UTC
56649,CALL0056697,ACC0024264,BRW0003381,2026-03-03 00:25:38,AGT0000632,CMP0000008,OUTBOUND,VND0000013,VOICEMAIL,489,UTC


In [105]:
cleaned_data["calls"] = calls_consolidated.copy()

print("Calls table updated successfully.")
print("Rows:", len(cleaned_data["calls"]))
print("Unique call IDs:", cleaned_data["calls"]["call_id"].nunique())

Calls table updated successfully.
Rows: 90011
Unique call IDs: 90000


In [106]:
# Inspect repeated agent IDs

agents_df = cleaned_data["agents"].copy()

print("Total rows:", len(agents_df))
print("Unique agent IDs:", agents_df["agent_id"].nunique())
print("Missing agent IDs:", agents_df["agent_id"].isna().sum())

print("\nColumn names:")
print(agents_df.columns.tolist())

print("\nSample records:")
display(agents_df.head(10))

Total rows: 30000
Unique agent IDs: 1000
Missing agent IDs: 0

Column names:
['agent_id', 'employee_code', 'agent_name', 'vendor_id', 'team', 'status', 'joined_at', 'updated_at']

Sample records:


,agent_id,employee_code,agent_name,vendor_id,team,status,joined_at,updated_at
0,AGT0000760,EMP00323,Vikram Shah,VND0000012,T3,INACTIVE,2025-06-19 20:30:53,2025-12-11 03:40:22
1,AGT0000171,EMP00079,Pooja Nair,VND0000012,T1,INACTIVE,2025-08-26 11:06:26,2025-10-11 08:30:33
2,AGT0000766,EMP00030,Sneha Das,VND0000002,FIELD,SUSPENDED,2024-05-30 11:23:06,2026-05-16 03:02:02
3,AGT0000031,EMP00334,Aarav Sharma,VND0000007,T1,INACTIVE,2024-10-11 18:58:28,2025-10-16 13:31:21
4,AGT0000669,EMP00014,Aarav Sharma,VND0000001,T3,SUSPENDED,2024-09-01 19:55:06,2026-07-04 03:33:11
5,AGT0000968,EMP00060,Amit Kumar,VND0000012,T1,ACTIVE,2025-04-30 13:05:13,2025-02-11 19:01:45
6,AGT0000155,EMP00636,Amit Kumar,VND0000007,DIGITAL,SUSPENDED,2024-12-07 13:58:39,2026-07-28 06:16:07
7,AGT0000480,EMP00420,Aarav Sharma,VND0000005,FIELD,INACTIVE,2024-02-16 22:56:23,2025-02-01 18:17:09
8,AGT0000404,EMP00148,Rahul Verma,VND0000009,DIGITAL,INACTIVE,2024-04-02 12:16:02,2025-10-13 14:12:16
9,AGT0000077,EMP00415,Aarav Sharma,VND0000003,T2,ACTIVE,2025-07-29 21:47:05,2026-05-28 01:38:51


In [107]:
# Investigate repeated agent IDs

agents_df = cleaned_data["agents"].copy()

# Count distinct values for each agent ID
agent_variations = (
    agents_df.groupby("agent_id")
    .agg(
        record_count=("agent_id", "size"),
        employee_codes=("employee_code", "nunique"),
        names=("agent_name", "nunique"),
        vendors=("vendor_id", "nunique"),
        teams=("team", "nunique"),
        statuses=("status", "nunique"),
        joined_dates=("joined_at", "nunique"),
        updated_dates=("updated_at", "nunique")
    )
)

print("Agent IDs:", len(agent_variations))
print("Agent IDs with multiple records:",
      (agent_variations["record_count"] > 1).sum())

print("\nVariation summary:")
display(agent_variations.describe())

print("\nSample agent IDs with multiple names, vendors, or employee codes:")
display(
    agent_variations[
        (agent_variations["employee_codes"] > 1) |
        (agent_variations["names"] > 1) |
        (agent_variations["vendors"] > 1)
    ].head(10)
)

Agent IDs: 1000
Agent IDs with multiple records: 1000

Variation summary:


,record_count,employee_codes,names,vendors,teams,statuses,joined_dates,updated_dates
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.0,1000.000000,1000.000000
mean,30.000000,29.604000,9.489000,13.006000,4.990000,3.0,30.000000,30.000000
std,5.570011,5.461044,0.717212,1.318978,0.099549,0.0,5.570011,5.570011
min,14.000000,14.000000,6.000000,7.000000,4.000000,3.0,14.000000,14.000000
25%,26.000000,26.000000,9.000000,12.000000,5.000000,3.0,26.000000,26.000000
50%,30.000000,29.000000,10.000000,13.000000,5.000000,3.0,30.000000,30.000000
75%,34.000000,33.000000,10.000000,14.000000,5.000000,3.0,34.000000,34.000000
max,48.000000,48.000000,10.000000,15.000000,5.000000,3.0,48.000000,48.000000



Sample agent IDs with multiple names, vendors, or employee codes:


,record_count,employee_codes,names,vendors,teams,statuses,joined_dates,updated_dates
agent_id,,,,,,,,
AGT0000001,23,23,10,10,5,3,23,23
AGT0000002,23,23,10,12,5,3,23,23
AGT0000003,28,28,10,12,5,3,28,28
AGT0000004,28,28,10,12,5,3,28,28
AGT0000005,29,29,10,15,5,3,29,29
AGT0000006,37,37,10,15,5,3,37,37
AGT0000007,30,29,10,15,5,3,30,30
AGT0000008,29,29,10,12,5,3,29,29
AGT0000009,25,24,10,12,5,3,25,25


In [108]:
# Inspect all records for one agent ID

sample_agent_id = "AGT0000001"

agent_history = (
    cleaned_data["agents"][
        cleaned_data["agents"]["agent_id"] == sample_agent_id
    ]
    .sort_values("updated_at")
)

display(agent_history)

print("Total records:", len(agent_history))
print("Distinct employee codes:", agent_history["employee_code"].nunique())
print("Distinct names:", agent_history["agent_name"].nunique())
print("Distinct vendors:", agent_history["vendor_id"].nunique())

,agent_id,employee_code,agent_name,vendor_id,team,status,joined_at,updated_at
9554,AGT0000001,EMP00883,Sneha Das,VND0000010,T3,INACTIVE,2025-09-15 12:18:43,2025-02-08 10:18:20
3016,AGT0000001,EMP00285,Sneha Das,VND0000013,DIGITAL,INACTIVE,2025-05-02 11:13:31,2025-02-14 12:38:08
18301,AGT0000001,EMP00191,Priya Mehta,VND0000015,FIELD,ACTIVE,2025-11-04 07:01:56,2025-03-09 16:24:54
21572,AGT0000001,EMP00745,Vikram Shah,VND0000008,T3,ACTIVE,2024-02-08 16:05:41,2025-03-09 22:26:39
8967,AGT0000001,EMP01082,Neha Singh,VND0000013,T1,INACTIVE,2025-05-02 01:36:54,2025-04-25 02:56:44
6520,AGT0000001,EMP00632,Ananya Rao,VND0000007,DIGITAL,ACTIVE,2024-08-02 00:51:56,2025-08-30 06:00:31
19963,AGT0000001,EMP00903,Sneha Das,VND0000002,T3,INACTIVE,2024-02-21 17:11:06,2025-09-18 08:17:45
9420,AGT0000001,EMP00723,Rahul Verma,VND0000013,T2,ACTIVE,2025-03-04 14:23:32,2025-10-11 06:30:44
25883,AGT0000001,EMP00404,Ananya Rao,VND0000014,T2,SUSPENDED,2025-11-13 07:54:11,2025-10-27 07:59:16
15127,AGT0000001,EMP00227,Sneha Das,VND0000007,T3,INACTIVE,2024-12-06 13:45:28,2025-11-26 03:28:09


Total records: 23
Distinct employee codes: 23
Distinct names: 10
Distinct vendors: 10


In [109]:
# Measure agent identity conflicts

agents_df = cleaned_data["agents"].copy()

identity_conflicts = (
    agents_df.groupby("agent_id")
    .agg(
        record_count=("agent_id", "size"),
        employee_codes=("employee_code", "nunique"),
        names=("agent_name", "nunique"),
        vendors=("vendor_id", "nunique")
    )
)

print("Total agent IDs:", len(identity_conflicts))

print(
    "Agent IDs with multiple employee codes:",
    (identity_conflicts["employee_codes"] > 1).sum()
)

print(
    "Agent IDs with multiple names:",
    (identity_conflicts["names"] > 1).sum()
)

print(
    "Agent IDs with multiple vendors:",
    (identity_conflicts["vendors"] > 1).sum()
)

print(
    "Agent IDs with all three conflicts:",
    (
        (identity_conflicts["employee_codes"] > 1) &
        (identity_conflicts["names"] > 1) &
        (identity_conflicts["vendors"] > 1)
    ).sum()
)

Total agent IDs: 1000
Agent IDs with multiple employee codes: 1000
Agent IDs with multiple names: 1000
Agent IDs with multiple vendors: 1000
Agent IDs with all three conflicts: 1000


In [110]:
borrowers_df = cleaned_data["borrowers"].copy()

print("Columns in borrowers table:")
print(borrowers_df.columns.tolist())

print("\nSample records:")
display(borrowers_df.head(5))

Columns in borrowers table:
['borrower_id', 'name', 'phone', 'email', 'city', 'created_at', 'updated_at', 'state']

Sample records:


,borrower_id,name,phone,email,city,created_at,updated_at,state
0,BRW0001072,Rahul Verma,9.037419e+09,user9183@example.com,Bhubaneswar,2025-09-25 08:02:29,2026-07-25 22:25:48,Odisha
1,BRW0009288,Neha Singh,9.154913e+09,user9581@example.com,Gurugram,2025-06-29 04:32:01,2025-03-04 12:15:30,Haryana
2,BRW0007855,Vikram Shah,9.986755e+09,user19407@example.com,Hyderabad,2025-11-23 20:30:50,2025-01-07 21:07:17,Telangana
3,BRW0005267,Priya Mehta,9.145160e+09,user4619@example.com,Jaipur,2026-05-09 11:34:12,2025-04-24 21:35:48,Rajasthan
4,BRW0005197,Priya Mehta,9.655567e+09,user4597@example.com,Mumbai,2025-02-13 03:53:20,2026-03-29 19:42:02,Maharashtra


In [111]:
# Investigate borrower identity conflicts

borrowers_df = cleaned_data["borrowers"].copy()

borrower_variations = (
    borrowers_df.groupby("borrower_id")
    .agg(
        record_count=("borrower_id", "size"),
        names=("name", "nunique"),
        phones=("phone", "nunique"),
        emails=("email", "nunique"),
        cities=("city", "nunique"),
        states=("state", "nunique")
    )
)

print("Total borrower records:", len(borrowers_df))
print("Unique borrower IDs:", borrowers_df["borrower_id"].nunique())

print(
    "Borrower IDs with multiple records:",
    (borrower_variations["record_count"] > 1).sum()
)
print(
    "Borrower IDs with multiple names:",
    (borrower_variations["names"] > 1).sum()
)
print(
    "Borrower IDs with multiple phone numbers:",
    (borrower_variations["phones"] > 1).sum()
)
print(
    "Borrower IDs with multiple email addresses:",
    (borrower_variations["emails"] > 1).sum()
)

print("\nSample conflicting borrower profiles:")
display(
    borrower_variations[
        (borrower_variations["names"] > 1) |
        (borrower_variations["phones"] > 1) |
        (borrower_variations["emails"] > 1)
    ].head(10)
)

Total borrower records: 30000
Unique borrower IDs: 11015
Borrower IDs with multiple records: 8518
Borrower IDs with multiple names: 8185
Borrower IDs with multiple phone numbers: 8399
Borrower IDs with multiple email addresses: 8334

Sample conflicting borrower profiles:


,record_count,names,phones,emails,cities,states
borrower_id,,,,,,
BRW0000001,3,2,3,3,2,2
BRW0000004,4,4,4,4,4,4
BRW0000005,2,2,2,2,2,1
BRW0000006,3,3,3,3,2,2
BRW0000007,4,3,4,4,4,4
BRW0000009,3,3,3,2,3,3
BRW0000012,5,4,5,5,4,4
BRW0000013,4,3,4,4,3,3
BRW0000015,2,2,2,2,2,2


In [112]:
# Check whether borrower timestamps are logically consistent

borrowers_df = cleaned_data["borrowers"].copy()

borrowers_df["created_at"] = pd.to_datetime(
    borrowers_df["created_at"], errors="coerce"
)

borrowers_df["updated_at"] = pd.to_datetime(
    borrowers_df["updated_at"], errors="coerce"
)

invalid_timestamps = borrowers_df[
    borrowers_df["updated_at"] < borrowers_df["created_at"]
]

print("Total borrower records:", len(borrowers_df))
print("Missing created_at:", borrowers_df["created_at"].isna().sum())
print("Missing updated_at:", borrowers_df["updated_at"].isna().sum())
print("Updated before created:", len(invalid_timestamps))
print(
    "Percentage with invalid timestamp order:",
    round(len(invalid_timestamps) / len(borrowers_df) * 100, 2),
    "%"
)

display(invalid_timestamps.head(5))

Total borrower records: 30000
Missing created_at: 0
Missing updated_at: 0
Updated before created: 15063
Percentage with invalid timestamp order: 50.21 %


,borrower_id,name,phone,email,city,created_at,updated_at,state
1,BRW0009288,Neha Singh,9.154913e+09,user9581@example.com,Gurugram,2025-06-29 04:32:01,2025-03-04 12:15:30,Haryana
2,BRW0007855,Vikram Shah,9.986755e+09,user19407@example.com,Hyderabad,2025-11-23 20:30:50,2025-01-07 21:07:17,Telangana
3,BRW0005267,Priya Mehta,9.145160e+09,user4619@example.com,Jaipur,2026-05-09 11:34:12,2025-04-24 21:35:48,Rajasthan
7,BRW0008369,Vikram Shah,9.737942e+09,user8941@example.com,Bhubaneswar,2026-05-27 08:50:59,2025-09-20 02:27:08,Odisha
8,BRW0002418,Sneha Das,9.608424e+09,user16450@example.com,Mumbai,2026-05-27 06:05:11,2025-07-19 15:06:52,Maharashtra


In [113]:
# Investigate repeated WhatsApp event IDs

whatsapp_df = cleaned_data["whatsapp_events"].copy()

print("Total rows:", len(whatsapp_df))
print("Unique event IDs:", whatsapp_df["whatsapp_event_id"].nunique())
print("Missing event IDs:", whatsapp_df["whatsapp_event_id"].isna().sum())

duplicate_whatsapp = whatsapp_df[
    whatsapp_df["whatsapp_event_id"].duplicated(keep=False)
].sort_values("whatsapp_event_id")

print("Rows with repeated event IDs:", len(duplicate_whatsapp))
print(
    "Number of repeated event IDs:",
    duplicate_whatsapp["whatsapp_event_id"].nunique()
)

display(duplicate_whatsapp.head(20))

Total rows: 60000
Unique event IDs: 60000
Missing event IDs: 0
Rows with repeated event IDs: 0
Number of repeated event IDs: 0


,whatsapp_event_id,account_id,borrower_id,event_at,message_id,event_type,template_code,provider_id


In [114]:
# Final exact-duplicate check across cleaned tables

for table_name, df in cleaned_data.items():
    duplicate_count = df.duplicated().sum()

    if duplicate_count > 0:
        print(f"{table_name}: {duplicate_count} exact duplicate rows remain")

print("Check complete.")

Check complete.


## 11. Final data-quality summary

Collect the major detected issues and treatment notes for the report. This summary is not a substitute for documenting unresolved limitations in the executive memo.

In [115]:
# Create a data-quality summary

quality_summary = pd.DataFrame([
    {
        "table": "borrowers",
        "issue": "Conflicting borrower profiles",
        "finding": "8,518 borrower IDs have multiple records",
        "action": "Preserve records; flag identity conflicts"
    },
    {
        "table": "borrowers",
        "issue": "Timestamp inconsistency",
        "finding": "50.21% have updated_at earlier than created_at",
        "action": "Preserve timestamps; flag invalid ordering"
    },
    {
        "table": "agents",
        "issue": "Conflicting agent identities",
        "finding": "All 1,000 agent IDs have multiple employee codes, names, and vendors",
        "action": "Preserve records; exclude from agent-level ranking"
    },
    {
        "table": "calls",
        "issue": "Repeated call IDs with conflicting timestamps",
        "finding": "11 call IDs have different timestamps",
        "action": "Preserve both records; investigate"
    },
    {
        "table": "payments",
        "issue": "Repeated payment IDs",
        "finding": "14 repeated IDs consolidated after core-field checks",
        "action": "Retain consolidation rule and audit trail"
    },
    {
        "table": "whatsapp_events",
        "issue": "Exact duplicate rows",
        "finding": "600 exact duplicates removed",
        "action": "Retain original source and cleaned table"
    }
])

display(quality_summary)

,table,issue,finding,action
0,borrowers,Conflicting borrower profiles,"8,518 borrower IDs have multiple records",Preserve records; flag identity conflicts
1,borrowers,Timestamp inconsistency,50.21% have updated_at earlier than created_at,Preserve timestamps; flag invalid ordering
2,agents,Conflicting agent identities,"All 1,000 agent IDs have multiple employee cod...",Preserve records; exclude from agent-level ran...
3,calls,Repeated call IDs with conflicting timestamps,11 call IDs have different timestamps,Preserve both records; investigate
4,payments,Repeated payment IDs,14 repeated IDs consolidated after core-field ...,Retain consolidation rule and audit trail
5,whatsapp_events,Exact duplicate rows,600 exact duplicates removed,Retain original source and cleaned table


In [116]:
quality_summary.to_csv(
    "data_quality_summary.csv",
    index=False
)

print("Data-quality summary saved successfully.")

Data-quality summary saved successfully.
